# MiniMax H3 Turbo LoRA：Colab 首幀／多參照圖片轉影片（含音訊）

這份 notebook 使用 ComfyUI 在 Colab 遠端 GPU 執行 MiniMax H3。網頁 notebook 預設使用 FL2VA：提供**一張首幀圖**與 `[Shot N]` 分鏡，生成含同步音訊的 MP4。本機 CLI 會切換成 Ref2VA，可提供 1–9 張語意參照圖和完整 prompt 文字檔。模型會依 CUDA 與 GPU 能力選擇 INT8、FP8 或 BF16 權重。Ref2VA 圖片依上傳順序標記為 `<Picture 1>`、`<Picture 2>` 等；它們是內容參照，不會自動固定到特定影片時間。

可直接在 Colab 網頁 notebook 使用，也可從本機終端透過 `run_colab_inference.sh` 執行，不必開啟 Colab notebook 網頁。CLI 模式會把本機參照圖與 prompt.txt 上傳到遠端 VM，完成後下載 MP4；模型運算仍在 Colab GPU，不是在 Mac 本機執行。首次執行需下載大型模型與 LoRA；建議高 RAM、A100 等級 GPU 和至少 60 GiB 可用磁碟；若 Ref2VA 選用 BF16 備援權重，需至少 80 GiB。免費 T4 通常不足。模型受 [MiniMax H3 授權](https://huggingface.co/MiniMaxAI/MiniMax-H3)約束。

從上到下執行各 cell：**安裝 → 下載模型 → 設定模式與參數 → 設定圖片 → 檢查 prompt → 推論 → 儲存 MP4**。網頁模式預設使用 FL2VA，會顯示預覽並提供下載；CLI 模式使用 Ref2VA，結果存到 `/content/minimax_h3_output.mp4` 供終端下載。

兩種模式都使用官方 Qwen3-VL 文字編碼器與雙 VAE 解碼。FL2VA 模式使用本 repo 的 v4 Turbo LoRA；Ref2VA 使用官方 ref2v 4-step LoRA 與對應模型。原 workflow 的 SageAttention、SolAttn、Spectrum、預覽與除錯節點為可略過的輔助節點；輸出可能與桌面 workflow 不完全相同。

參考：[MiniMax 官方模型卡](https://huggingface.co/MiniMaxAI/MiniMax-H3)、[官方 Ref2VA prompt 指南](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/docs/VIDEO_PROMPT_WRITING_GUIDE_ref_en.md)、[FL2VA prompt 指南](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/docs/VIDEO_PROMPT_WRITING_GUIDE_base_en.md)、[ComfyUI H3 節點](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy_extras/nodes_minimax_h3.py)、[原 workflow](https://huggingface.co/drbaph/MiniMax-H3-Turbo-Lora-ComfyUI/blob/main/fl_minimax_h3_turbo_lora_example_workflow.json)。


## 生成時間：最短、最長與實際秒數

| 來源／用途 | 最短 | 最長 | 這份 notebook 的處理 |
| --- | ---: | ---: | --- |
| [MiniMax H3 模型卡](https://huggingface.co/MiniMaxAI/MiniMax-H3)的輸出規格 | 4 秒 | 15 秒 | 接受 `DURATION_SECONDS` 介於 4–15 |
| [ComfyUI H3 節點](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy_extras/nodes_minimax_h3.py)標示的約略已訓練範圍 | 約 5 秒（124 幀） | 約 15 秒（362 幀） | **建議使用 5–15 秒**；4 秒屬較不穩定的邊界設定 |

H3 以 **24 fps** 生成，ComfyUI 將要求幀數向上對齊到 `17k+5`。所以 `DURATION_SECONDS` 是**要求秒數**，不一定等於 MP4 的精確秒數。常見例子：4 → **107 幀 / 4.46 秒**；5 → **124 幀 / 5.17 秒**；12 → **294 幀 / 12.25 秒**；15 → **362 幀 / 15.08 秒**。若成品必須**嚴格不超過 15.00 秒**，在 notebook 設 `DURATION_SECONDS = 14.375`，對應 345 幀 / 14.375 秒；也可以在輸出後另行裁切，但會改變結尾音訊。

多鏡頭的切換時間要以影片**起點**為 0 秒、嚴格遞增、落在要求秒數之內。例如 12 秒影片可在 4.0 與 8.0 秒切成三個 shots。這些是 prompt 中希望模型切換的時間，不是剪輯軟體的硬切點。

ComfyUI 節點的數值欄位允許更短或更長的幀數，但那是介面範圍，**不代表模型在該範圍內經過訓練或保證品質**。


## 1. 檢查 Colab 環境並安裝 ComfyUI

本機 CLI 會自動建立遠端 GPU runtime；用 Colab 網頁時，請先在「執行階段 → 變更執行階段類型」選 GPU。下一個 cell 在 `/content` 安裝 ComfyUI 和影片輸出節點；需有 `ffmpeg`、`ffprobe` 與至少 60 GiB 可用磁碟；Ref2VA 的 BF16 備援模式需要 80 GiB。


In [ ]:
import os, shutil, subprocess, sys, torch
from pathlib import Path

assert torch.cuda.is_available(), "請在 Colab「執行階段 > 變更執行階段類型」選擇 GPU。"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
VRAM_GIB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("PyTorch CUDA:", torch.version.cuda)
print("Free disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
INFERENCE_MODE = os.environ.get("H3_INFERENCE_MODE", "reference").strip().lower()
assert INFERENCE_MODE in {"first_frame", "reference", "r2v", "t2v", "text_to_video", "director", "char_swap", "v2v", "longshot", "continuum", "ai_upscale", "face_refine"}, f"H3_INFERENCE_MODE không hợp lệ: {INFERENCE_MODE}"
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
CUDA_MAJOR = int((torch.version.cuda or "0").split(".", 1)[0])
if CUDA_MAJOR >= 13 and GPU_CAPABILITY >= (8, 0):
    REF2VA_VARIANT = "int8_convrot"
elif GPU_CAPABILITY >= (8, 9):
    REF2VA_VARIANT = "fp8_scaled"
else:
    REF2VA_VARIANT = "bf16"
MINIMUM_FREE_DISK = 80 * 2**30 if REF2VA_VARIANT == "bf16" else 60 * 2**30
assert shutil.disk_usage("/content").free >= MINIMUM_FREE_DISK, (
    f"目前選用 {REF2VA_VARIANT or 'FL2VA'} 模型，至少需要 {MINIMUM_FREE_DISK / 2**30:.0f} GiB 可用磁碟。"
)
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "需要 ffmpeg 與 ffprobe 才能輸出並檢查 MP4 音訊。"
COMFY = Path("/content/ComfyUI")
VHS = COMFY / "custom_nodes" / "ComfyUI-VideoHelperSuite"
DIRECTOR = COMFY / "custom_nodes" / "ComfyUI_MiniMaxH3_Director"
PROMPT_COMPILER = COMFY / "custom_nodes" / "H3_Prompt_Compiler"
H3_LONGSHOT = COMFY / "custom_nodes" / "H3_Longshot"
H3_CONTINUUM = COMFY / "custom_nodes" / "ComfyUI-H3-Continuum"
H3_UPSAMPLER = COMFY / "custom_nodes" / "ComfyUI-H3-Video-Upsampler"
H3_YCNODES = COMFY / "custom_nodes" / "ComfyUI-YCNodes-MiniMax-H3"
H3_SELFLIFT_NODE = COMFY / "custom_nodes" / "comfyui-SelfLift"
H3_KJNODES = COMFY / "custom_nodes" / "ComfyUI-KJNodes"
H3_DETAIL_CONTRAST = COMFY / "custom_nodes" / "comfyui-image-detail-contrast"
H3_ORION4D = COMFY / "custom_nodes" / "Orion4D_FXMax"
H3_ONECLICK_SWAP = COMFY / "custom_nodes" / "comfyui-h3-oneclick-swap"
H3_FANTASTIC = COMFY / "custom_nodes" / "ComfyUI-Fantastic-MiniMaxH3-PromptBuilder"
H3_CONTEXT_LOOP = COMFY / "custom_nodes" / "ComfyUI-MiniMaxH3-Context-Loop"
H3_FACEREFINE = COMFY / "custom_nodes" / "ComfyUI-H3-FaceRefine"
SETUP_MARKER = Path("/content/.h3_comfyui_setup_v1")
if SETUP_MARKER.exists():
    assert COMFY.is_dir() and VHS.is_dir(), "Colab setup marker exists, but ComfyUI files are missing; remove /content/.h3_comfyui_setup_v1 and retry."
    if not (COMFY / "comfy_extras" / "nodes_frame_interp.py").is_file():
        subprocess.run(["git", "-C", str(COMFY), "pull", "--ff-only"], check=False)
    print("重用本次 Colab session 已安裝的 ComfyUI 環境")
else:
    if not COMFY.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"), "huggingface_hub", "hf_xet", "hf_transfer", "pillow", "requests"], check=True)
    if not VHS.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", str(VHS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(VHS / "requirements.txt")], check=True)
    if not DIRECTOR.exists():
        try:
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git", str(DIRECTOR)], check=False)
        except Exception as e:
            print("Director node optional clone warning:", e)
    SETUP_MARKER.touch()
print("ComfyUI:", COMFY)
for _repo_url, _dest in [
    ("https://github.com/r34vtraining/H3_Prompt_Compiler.git", PROMPT_COMPILER),
    ("https://github.com/r34vtraining/H3_Longshot.git", H3_LONGSHOT),
    ("https://github.com/ukr8b3g-cmyk/ComfyUI-H3-Continuum.git", H3_CONTINUUM),
    ("https://github.com/dntpi/ComfyUI-H3-Video-Upsampler.git", H3_UPSAMPLER),
    ("https://github.com/yichengup/ComfyUI-YCNodes-MiniMax-H3.git", H3_YCNODES),
    ("https://github.com/facok/comfyui-SelfLift.git", H3_SELFLIFT_NODE),
    ("https://github.com/kijai/ComfyUI-KJNodes.git", H3_KJNODES),
    ("https://github.com/D3voz/comfyui-image-detail-contrast.git", H3_DETAIL_CONTRAST),
    ("https://github.com/orion4d/Orion4D_FXMax.git", H3_ORION4D),
    ("https://github.com/D3voz/comfyui-h3-oneclick-swap.git", H3_ONECLICK_SWAP),
    ("https://github.com/Adudeguyman/ComfyUI-Fantastic-MiniMaxH3-PromptBuilder.git", H3_FANTASTIC),
    ("https://github.com/ethanfel/ComfyUI-MiniMaxH3-Context-Loop.git", H3_CONTEXT_LOOP),
    ("https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git", H3_FACEREFINE),
]:
    if not _dest.exists():
        try:
            subprocess.run(["git", "clone", "--depth", "1", _repo_url, str(_dest)], check=True)
            _req = _dest / "requirements.txt"
            if _req.is_file():
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)], check=False)
        except Exception as _e:
            print("Custom node clone warning:", _e)
_ve_py = H3_FANTASTIC / "video_edit.py"
if _ve_py.is_file():
    _ve_txt = _ve_py.read_text(encoding="utf-8")
    if 'raise ValueError("The mask is empty on every frame' in _ve_txt:
        _ve_txt = _ve_txt.replace(
            'raise ValueError("The mask is empty on every frame, so nothing would change.")',
            'obj = torch.ones_like(obj); print("[MiniMaxH3FantasticVideoEditLatent] SAM mask empty -> fallback to full-frame mask")'
        )
        _ve_py.write_text(_ve_txt, encoding="utf-8")
if INFERENCE_MODE == "face_refine" or os.environ.get("H3_FACE_REFINE", "0").strip().lower() in ("1", "true", "yes"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "scipy", "scenedetect>=0.7"], check=False)
# Auto-Idle Watchdog on remote Colab VM: self-terminate after 15 mins of inactivity to save CU
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
_ACTIVITY_FILE = Path("/content/.h3_last_activity")
_ACTIVITY_FILE.write_text(str(__import__("time").time()))
if not Path("/content/.h3_watchdog_started").exists():
    Path("/content/.h3_watchdog_started").touch()
    subprocess.Popen([
        sys.executable, "-c",
        "import os, time, pathlib, requests\n"
        "act = pathlib.Path('/content/.h3_last_activity')\n"
        "while True:\n"
        "    time.sleep(30)\n"
        "    try:\n"
        "        q = requests.get('http://127.0.0.1:8188/queue', timeout=3).json()\n"
        "        if q.get('queue_running') or q.get('queue_pending'):\n"
        "            act.write_text(str(time.time()))\n"
        "            continue\n"
        "    except Exception:\n"
        "        pass\n"
        "    last = float(act.read_text()) if act.exists() else time.time()\n"
        "    if time.time() - last > 900:\n"
        "        os.system('kill -9 1')\n"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)


## 2. 下載模型

`INFERENCE_MODE` 在環境檢查 cell 預設為 `first_frame`；本機 CLI 會自動設為 `reference`。網頁模式可在環境檢查 cell 選擇 `reference`。Ref2VA 會依 CUDA 版本與 GPU compute capability 選用官方 INT8 ConvRot、FP8 scaled 或 BF16 diffusion 權重，並配對 video VAE 與 4-step Turbo LoRA；BF16 備援需至少 80 GiB 可用磁碟。FL2VA 仍可調 `DIFFUSION_VARIANT`。文字編碼器和 audio VAE 共用。首次執行需要下載大型模型；Colab 執行階段重建後可能需要重新下載。


In [ ]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

BASE_REPO = "Comfy-Org/MiniMax-H3"
TEXT_ENCODER = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
AUDIO_VAE = "minimax_h3_audio_vae_fp32.safetensors"

LORA_CHOICE = os.environ.get("H3_LORA", "pdmd").strip().lower()
SWAP_LORA = None
LMS_SHARP_LORA = None
USE_LMS_SHARP = (LORA_CHOICE == "lms_sharp" or os.environ.get("H3_LMS_SHARP", "1").strip().lower() not in ("0", "false", "no"))
LMS_SHARP_REPO = "Alissonerdx/Minimax-H3-ComfyUI"
LMS_SHARP_REMOTE = "loras/minimax_h3_lms_v1.0_r64.safetensors"
if USE_LMS_SHARP:
    LMS_SHARP_LORA = Path(LMS_SHARP_REMOTE).name

USE_MATLOW_FUSED = (REF2VA_VARIANT == "int8_convrot" and LORA_CHOICE not in ("fasth3_omni", "fasth3_t2v", "fasth3_trim"))
MATLOW_REPO = "MATLOWAI/minimax-h3-fused-turbo-int8-convrot"
MATLOW_REMOTE = "diffusion_models/minimax_h3_fused_refdelta_r1024_turbo8_mystic07_int8_convrot.safetensors"

if INFERENCE_MODE in ("reference", "r2v", "t2v", "text_to_video", "director", "char_swap", "v2v", "longshot", "continuum", "ai_upscale", "face_refine"):
    DIFFUSION = Path(MATLOW_REMOTE).name if USE_MATLOW_FUSED else f"minimax_h3_ref2va_pruned_{REF2VA_VARIANT}.safetensors"
    VIDEO_VAE = "minimax_h3_video_vae_fp16.safetensors"
    if INFERENCE_MODE in ("char_swap", "v2v") or LORA_CHOICE == "char_swap":
        if LORA_CHOICE == "mystic":
            LORA_REPO = "speedyrulz/mini"
            LORA_REMOTE = "MysticXXX_MMH3-V4-ref2va.safetensors"
        elif USE_MATLOW_FUSED:
            LORA_REPO = None
            LORA_REMOTE = None
        else:
            LORA_REPO = "Iwannapose/minimax_h3_pdmd_4nfe_comfyui"
            LORA_REMOTE = "minimax_h3_pdmd_4nfe_comfyui_v6.safetensors"
        SWAP_LORA_REPO = "Alissonerdx/Minimax-H3-ComfyUI"
        SWAP_LORA_REMOTE = "loras/minimax_h3_character_swap_v1.1_ref2va_fro.safetensors"
        SWAP_LORA = Path(SWAP_LORA_REMOTE).name
    elif LORA_CHOICE == "mystic":
        LORA_REPO = "speedyrulz/mini"
        LORA_REMOTE = "MysticXXX_MMH3-V4-ref2va.safetensors"
    elif USE_MATLOW_FUSED or LORA_CHOICE in ("fasth3_omni", "fasth3_t2v", "fasth3_trim"):
        LORA_REPO = None
        LORA_REMOTE = None
    elif LORA_CHOICE in ("pdmd", "lms_sharp"):
        LORA_REPO = "Iwannapose/minimax_h3_pdmd_4nfe_comfyui"
        LORA_REMOTE = "minimax_h3_pdmd_4nfe_comfyui_v6.safetensors"
    else:
        LORA_REPO = BASE_REPO
        LORA_REMOTE = "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
    print(f"Variant: {REF2VA_VARIANT} (Fused={USE_MATLOW_FUSED}) | Mode: {INFERENCE_MODE} | LoRA Engine: {LORA_CHOICE} ({LORA_REMOTE}) | Swap LoRA: {SWAP_LORA} | LMS Sharp 2K: {LMS_SHARP_LORA}")
else:
    VIDEO_VAE = "minimax_h3_video_vae_fp16.safetensors"
    if USE_MATLOW_FUSED:
        DIFFUSION = Path(MATLOW_REMOTE).name
        LORA_REPO = None
        LORA_REMOTE = None
    else:
        LORA_REPO = "lightx2v/Minimax-h3-Turbo"
        DIFFUSION_VARIANT = "auto"
        assert DIFFUSION_VARIANT in {"auto", "fp8_scaled", "int8_convrot"}
        use_int8 = DIFFUSION_VARIANT == "int8_convrot" or (
            DIFFUSION_VARIANT == "auto" and (torch.version.cuda or "").startswith("13."))
        DIFFUSION = ("minimax_h3_fl2va_pruned_int8_convrot.safetensors" if use_int8
                     else "minimax_h3_fl2va_pruned_fp8_scaled.safetensors")
        LORA_REMOTE = "minimax_h3_fl2v_turbo_4step_v1.3_768p_comfyui_bf16.safetensors"

if LORA_CHOICE == "fasth3_omni":
    if REF2VA_VARIANT == "int8_convrot":
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastH3-Omni-Comfy", "diffusion_models/fasth3_omniref_8step_pruned_int8_convrot.safetensors"
    elif REF2VA_VARIANT == "fp8_scaled":
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastH3-Omni-FP8-Comfy", "fasth3_omniref_fp8_comfy.safetensors"
    else:
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastH3-Omni-Comfy", "diffusion_models/fasth3_omniref_8step_pruned_bf16.safetensors"
    DIFFUSION = Path(DIFF_REMOTE).name
elif LORA_CHOICE == "fasth3_t2v":
    if REF2VA_VARIANT == "int8_convrot":
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastVideo-FastH3-Comfy", "diffusion_models/fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors"
    elif REF2VA_VARIANT == "fp8_scaled":
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastVideo-FastH3-Trim-Comfy", "diffusion_models/fastvideo_fasth3_trim_8step_fp8.safetensors"
    else:
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastVideo-FastH3-Comfy", "diffusion_models/fastvideo_fasth3_8step_v2_pruned_bf16.safetensors"
    DIFFUSION = Path(DIFF_REMOTE).name
elif LORA_CHOICE == "fasth3_trim":
    if REF2VA_VARIANT == "int8_convrot":
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastVideo-FastH3-Trim-Comfy", "diffusion_models/fastvideo_fasth3_trim_8step_int8_convrot.safetensors"
    elif REF2VA_VARIANT == "fp8_scaled":
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastVideo-FastH3-Trim-Comfy", "diffusion_models/fastvideo_fasth3_trim_8step_fp8.safetensors"
    else:
        DIFF_REPO, DIFF_REMOTE = "FastVideo/FastVideo-FastH3-Trim-Comfy", "diffusion_models/fastvideo_fasth3_trim_8step_bf16.safetensors"
    DIFFUSION = Path(DIFF_REMOTE).name
elif USE_MATLOW_FUSED:
    DIFF_REPO, DIFF_REMOTE = MATLOW_REPO, MATLOW_REMOTE
else:
    DIFF_REPO, DIFF_REMOTE = BASE_REPO, "diffusion_models/" + DIFFUSION

model_files = [
    (DIFF_REPO, DIFF_REMOTE, "diffusion_models"),
    (BASE_REPO, "text_encoders/" + TEXT_ENCODER, "text_encoders"),
    (BASE_REPO, "vae/" + VIDEO_VAE, "vae"),
    (BASE_REPO, "vae/" + AUDIO_VAE, "vae"),
]
if LORA_REPO and LORA_REMOTE:
    model_files.append((LORA_REPO, LORA_REMOTE, "loras"))
if SWAP_LORA:
    model_files.append((SWAP_LORA_REPO, SWAP_LORA_REMOTE, "loras"))
if LMS_SHARP_LORA:
    model_files.append((LMS_SHARP_REPO, LMS_SHARP_REMOTE, "loras"))

LORA = Path(LORA_REMOTE).name if LORA_REMOTE else None
for repo, remote_name, folder in model_files:
    target = COMFY / "models" / folder / Path(remote_name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file() or target.stat().st_size == 0:
        dl_dir = COMFY / "models" if ("/" in remote_name and remote_name.split("/", 1)[0] == folder) else COMFY / "models" / folder
        downloaded = Path(hf_hub_download(repo_id=repo, filename=remote_name, local_dir=dl_dir))
        if downloaded != target and downloaded.is_file():
            shutil.move(str(downloaded), str(target))
        assert target.is_file(), f"下載路徑不符：{downloaded} != {target}"
    print(folder, target.name, f"{target.stat().st_size / 2**30:.2f} GiB")

# Tải RIFE 48fps (chỉ mất 10s nội suy); chỉ chạy H3VideoUpsampler (tốn +4 phút) khi H3_AI_UPSCALE=1
AI_UPSCALE = os.environ.get("H3_AI_UPSCALE", "0").strip().lower() in ("1", "true", "yes")
USE_MASKED_SWAP = (INFERENCE_MODE in ("char_swap", "v2v") and os.environ.get("H3_MASKED_SWAP", "1").strip().lower() not in ("0", "false", "no"))
USE_FACE_REFINE = (INFERENCE_MODE == "face_refine" or os.environ.get("H3_FACE_REFINE", "0").strip().lower() in ("1", "true", "yes"))
USE_COLOR_GRADE = (os.environ.get("H3_COLOR_GRADE", "1").strip().lower() not in ("0", "false", "no"))
USE_SELFLIFT = (os.environ.get("H3_SELFLIFT", "1").strip().lower() not in ("0", "false", "no") and not USE_MASKED_SWAP and INFERENCE_MODE != "face_refine")
_extra_models = [
    ("Comfy-Org/Frame-Interpolation", "frame_interpolation/rife_v4.26_heavy.safetensors", "frame_interpolation", "rife_v4.26_heavy.safetensors"),
]
if AI_UPSCALE or USE_SELFLIFT:
    _extra_models.append(
        ("Alissonerdx/Minimax-H3-ComfyUI", "latent_upscaler/h3_upscaler_lms_v0.1.safetensors", "latent_upscale_models", "minimax_h3_latent_upscaler_3d_bf16.safetensors")
    )
if AI_UPSCALE:
    _extra_models.append(
        ("sandpies/ComfyUI-H3-Video-Upsampler", "lora/h3upscale_v2_x2.safetensors", "loras", "h3upscale_v2_x2.safetensors")
    )
if USE_MASKED_SWAP:
    _extra_models.append(
        ("Comfy-Org/sam3", "checkpoints/sam3.1_multiplex_fp16.safetensors", "checkpoints", "sam3.1_multiplex_fp16.safetensors")
    )
if USE_FACE_REFINE:
    _extra_models.extend([
        ("Bingsu/adetailer", "face_yolov8m.pt", "ultralytics/bbox", "face_yolov8m.pt"),
        ("Bingsu/adetailer", "person_yolov8m-seg.pt", "ultralytics/segm", "person_yolov8m-seg.pt"),
    ])
if True:
    for _u_repo, _u_rem, _u_dir, _u_name in _extra_models:
        _ut = COMFY / "models" / _u_dir / _u_name
        _ut.parent.mkdir(parents=True, exist_ok=True)
        if not _ut.is_file() or _ut.stat().st_size == 0:
            _dl = Path(hf_hub_download(repo_id=_u_repo, filename=_u_rem, local_dir=_ut.parent))
            if _dl != _ut and _dl.is_file():
                shutil.move(str(_dl), str(_ut))
        print(_u_dir, _ut.name, f"{_ut.stat().st_size / 2**20:.1f} MiB")
        if _u_dir.startswith("ultralytics/"):
            _ut_root = COMFY / "models" / "ultralytics" / _u_name
            if not _ut_root.is_file():
                shutil.copy2(_ut, _ut_root)


## 3. 設定多個 shots 與可調參數

先修改下一個程式 cell。`SHOTS` 清單就是分鏡表：每一項的 `start_seconds` 是**鏡頭開始時間**，`description` 寫畫面、角色動作、鏡頭運動，以及該 shot 中出現的對白或可聽見的動作聲。第一個 shot 固定 0 秒；後續 shot 會自動寫成官方格式：`[Shot 2] At 00:04.000, the camera cuts to ...`。要新增鏡頭，複製清單內一筆並修改開始時間及描述；要只有一鏡，保留第一筆即可。

**prompt 怎麼寫：**Shot 1 先描述上傳圖片 `<Picture 1>` 中既有的人物、場景、衣物與構圖，再寫動作。後續 shot 延續相同角色，明確說明新景別或新視角、切鏡後的動作。鏡頭運動可用「push in / pan right / tracking shot」加上幅度與速度。若有人說話，把對白放在該 shot，例如 `The woman (S1) says: <d>[Chinese] 你好。</d>`；同一人跨 shot 保持相同 `(S1)`。`OVERALL_SOUNDSCAPE` 寫全片環境與動作聲；`NON_DIEGETIC_MUSIC` 寫觀眾能聽、場內人物聽不到的配樂，沒有就填 `N/A`。對白不要再抄進 soundscape。官方提示詞結構見[寫作指南](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/docs/VIDEO_PROMPT_WRITING_GUIDE_base_en.md)。

| 設定 | 預設 | 用途與建議 |
| --- | --- | --- |
| `WIDTH`, `HEIGHT` | 1376 × 768 | 原 workflow 16:9、約 1 MP；兩者需為 32 的倍數。降低尺寸可減少記憶體與時間，但畫質會下降。H3 原生通常以短邊 768px 為目標。 |
| `DURATION_SECONDS` | 12 | 整支影片的要求長度，允許 4–15 秒，建議 5–15 秒；對齊後實際秒數見上一節。 |
| `SHOTS` | 三鏡 | 同一次生成中的分鏡與切鏡時間；不是三次模型推論。模型對切鏡的遵循屬生成行為，不能保證逐幀精準。 |
| `OVERALL_SOUNDSCAPE`, `NON_DIEGETIC_MUSIC` | 雨林環境聲、無配樂 | 分別控制環境／動作聲與觀眾聽到的配樂；在 shots 的描述中寫對白與場景內音樂。 |
| `SEED` | 原 workflow 固定值 | 改變起始噪聲；相同 seed 與參數有助比較，但不保證跨版本得到完全相同影片。 |
| `STEPS` | 4（Ref2VA）／8（FL2VA） | Ref2VA 官方 Turbo LoRA 使用 4 步；FL2VA 的 v4 Turbo LoRA 原 workflow 使用 6–8 步。增加步數會更慢，效果不一定更好。 |
| `LORA_STRENGTH` | 1.0 | LoRA 影響程度；可先試 0.8–1.2。0 表示不用 LoRA，可能需要較多採樣步數。 |
| `SAMPLER_NAME`, `SCHEDULER_NAME` | `euler`, `beta` | 原 workflow 的取樣器與時間排程；改動可能同時影響影片與音訊品質。 |
| `DENOISE` | 1.0 | 對空白 latent 做完整生成；此流程建議維持 1.0。 |
| `VIDEO_CRF` | 20 | MP4 壓縮品質；數字越低通常檔案越大、畫面越清楚，H.264 可用 0–51。只影響輸出壓縮，不影響模型生成。 |
| `OUTPUT_PREFIX` | `MiniMax_H3_Colab` | 輸出檔名前綴。 |
| `FIRST_FRAME_MEGAPIXELS`, `FIRST_FRAME_RESOLUTION_STEPS`, `FIRST_FRAME_UPSCALE_METHOD` | `1.0`, `32`, `nearest-exact` | 僅 FL2VA 使用：送入 H3 前縮放首幀；Ref2VA 使用 `REFERENCE_IMAGE_SIZE` 另行處理參照圖。 |
| `LOW_VRAM_MODE` | `auto` | `auto` 在 GPU VRAM 少於 48 GiB 時啟用 ComfyUI `--lowvram`；`on` 強制啟用，`off` 關閉。省 VRAM 通常會更慢；更改後請重新啟動 Colab 執行階段，再由上往下執行。 |
| `PROMPT_OVERRIDE` | 空字串 | 填入完整 prompt 時直接使用。Ref2VA 模式需使用官方參照格式；CLI 會優先讀取 `--prompt` 指定的 UTF-8 文字檔。 |

**提示詞組裝的界線：**FL2VA 會把 shot 與聲音欄位排成官方格式，**不會自動改寫或潤飾內容**；Ref2VA 直接使用完整 prompt。官方 H3-Context-IR 是另一套未隨開源權重提供的前處理服務。

**圖片模式：**`first_frame` 使用 FL2VA 與單張首幀；`reference` 使用 Ref2VA，接受最多 9 張語意參照圖片，依序對應 `<Picture 1>` 至 `<Picture 9>`。參照圖不會自動成為不同 shot 的時間軸 keyframe。H3 以 24 fps 輸出；影片與音訊各用自己的 VAE。


In [ ]:
import math

WIDTH = int(os.environ.get("H3_WIDTH", "1024"))
HEIGHT = int(os.environ.get("H3_HEIGHT", "576"))
if WIDTH in (1376, 1344) and HEIGHT == 768 and os.environ.get("H3_FAST_R1024", "1") != "0":
    WIDTH, HEIGHT = 1024, 576
elif WIDTH == 768 and HEIGHT in (1376, 1344) and os.environ.get("H3_FAST_R1024", "1") != "0":
    WIDTH, HEIGHT = 576, 1024
# Keep full native 1344x768 (1.0 MP) H3 training resolution for maximum sharpness
DURATION_SECONDS = float(os.environ.get("H3_DURATION_SECONDS", "12.0"))
if LORA_CHOICE == "fasth3_omni":
    STEPS = 8
    SAMPLER_NAME = "euler"
    SCHEDULER_NAME = "simple"
elif LORA_CHOICE in ("fasth3_t2v", "fasth3_trim"):
    STEPS = 8
    SAMPLER_NAME = "res_multistep"
    SCHEDULER_NAME = "simple"
elif globals().get("USE_MATLOW_FUSED", False):
    STEPS = 4
    SAMPLER_NAME = "res_multistep"
    SCHEDULER_NAME = "simple"
elif INFERENCE_MODE in ("char_swap", "v2v") or LORA_CHOICE == "char_swap":
    STEPS = 4
    SAMPLER_NAME = "euler"
    SCHEDULER_NAME = "beta"
elif INFERENCE_MODE in ("reference", "r2v", "t2v", "text_to_video", "director", "longshot", "continuum"):
    STEPS = 4
    SAMPLER_NAME = "euler"
    SCHEDULER_NAME = "beta"
else:
    STEPS = 4
    SAMPLER_NAME = "euler"
    SCHEDULER_NAME = "beta"
USE_MASKED_SWAP = (INFERENCE_MODE in ("char_swap", "v2v") and os.environ.get("H3_MASKED_SWAP", "1").strip().lower() not in ("0", "false", "no"))
USE_FACE_REFINE = (INFERENCE_MODE == "face_refine" or os.environ.get("H3_FACE_REFINE", "0").strip().lower() in ("1", "true", "yes"))
USE_COLOR_GRADE = (os.environ.get("H3_COLOR_GRADE", "1").strip().lower() not in ("0", "false", "yes" if False else "no"))
SWAP_EDIT_MODE = os.environ.get("H3_SWAP_EDIT_MODE", "person").strip().lower() or "person"
SWAP_TARGET = os.environ.get("H3_SWAP_TARGET", "person").strip() or "person"
SWAP_HAT_FIX = os.environ.get("H3_SWAP_HAT_FIX", "0").strip().lower() in ("1", "true", "yes")
SWAP_NO_LIPSYNC = os.environ.get("H3_SWAP_NO_LIPSYNC", "0").strip().lower() in ("1", "true", "yes")
USE_SELFLIFT = (os.environ.get("H3_SELFLIFT", "1").strip().lower() not in ("0", "false", "no") and not USE_MASKED_SWAP and INFERENCE_MODE != "face_refine")
if USE_SELFLIFT and INFERENCE_MODE not in ("longshot", "continuum", "ai_upscale", "face_refine"):
    SAMPLER_NAME = "euler"
    SCHEDULER_NAME = "beta"
    STEPS = int(os.environ.get("H3_SELFLIFT_STEPS", "8"))
    SELFLIFT_TRANSITION_STEP = max(1, min(STEPS - 1, int(round(STEPS * 0.75))))
    print(f"⚡ [SelfLift Progressive 4K] Bật lấy mẫu phân giải lũy tiến: {SELFLIFT_TRANSITION_STEP}/{STEPS} bước ở 0.5x Low-Res + 3D Latent Lift + {STEPS - SELFLIFT_TRANSITION_STEP} bước Full-Res!")
SEED = int(os.environ.get("H3_SEED", "768914106098754"))
LORA_STRENGTH = 1.0
DENOISE = 1.0
VIDEO_CRF = int(os.environ.get("H3_CRF", "15"))
OUTPUT_PREFIX = os.environ.get("H3_OUTPUT_PREFIX", "MiniMax_H3_Colab")
REFERENCE_IMAGE_SIZE = "match" if INFERENCE_MODE in ("char_swap", "v2v") else os.environ.get("H3_REF_SIZE", "match")
FIRST_FRAME_MEGAPIXELS = 1.0
FIRST_FRAME_RESOLUTION_STEPS = 32
FIRST_FRAME_UPSCALE_METHOD = "lanczos"
LOW_VRAM_MODE = "auto"

SHOTS = [
    {
        "start_seconds": 0.0,
        "description": (
            "Photorealistic wildlife documentary. The chameleon in <Picture 1> "
            "remains on the same wet moss-covered branch, preserving its shape, "
            "large turreted eye, textured scales and rainforest background."
        ),
    },
]
OVERALL_SOUNDSCAPE = "Quiet natural room ambience."
NON_DIEGETIC_MUSIC = "N/A"
PROMPT_OVERRIDE = ""

assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, "寬高必須為正整數且是 32 的倍數。"
assert isinstance(DURATION_SECONDS, (int, float)) and math.isfinite(DURATION_SECONDS) and 2.0 <= DURATION_SECONDS <= (480.0 if INFERENCE_MODE in ("longshot", "continuum") else 15.0), "請將影片長度設為 2–15 秒。"
assert isinstance(STEPS, int) and STEPS > 0, "STEPS 必須是正整數。"
assert isinstance(SEED, int) and 0 <= SEED < 2**64, "SEED 必須是 0 到 2**64-1 的整數。"
assert math.isfinite(LORA_STRENGTH) and 0 <= LORA_STRENGTH <= 2, "LoRA 強度請設為 0–2。"
assert 0 < DENOISE <= 1, "DENOISE 請設為 0–1 之間（不含 0）。"
assert isinstance(VIDEO_CRF, int) and 0 <= VIDEO_CRF <= 51, "VIDEO_CRF 請設為 0–51 的整數。"
assert isinstance(OUTPUT_PREFIX, str) and OUTPUT_PREFIX.strip(), "請設定輸出檔名前綴。"
assert REFERENCE_IMAGE_SIZE in {"match", "max"}, "REFERENCE_IMAGE_SIZE 請使用 match 或 max。"
assert LOW_VRAM_MODE in {"auto", "on", "off"}, "LOW_VRAM_MODE 請使用 auto、on 或 off。"

requested_frames = max(5, round(DURATION_SECONDS * 24))
FRAMES = requested_frames + (5 - requested_frames % 17) % 17
ACTUAL_SECONDS = FRAMES / 24
print(f"模式：{INFERENCE_MODE}；要求長度：{DURATION_SECONDS:.3f} 秒；排程：{FRAMES} 幀 / {ACTUAL_SECONDS:.3f} 秒（24 fps）| Sampler: {SAMPLER_NAME} ({STEPS} steps)")


### 3.1 組裝並預覽模型真正收到的 prompt

FL2VA 模式會依照官方首幀格式組裝 `SHOTS`、環境聲與配樂。Ref2VA 模式則直接使用完整自訂 prompt；CLI 會從本機 `--prompt` 檔案讀取原文，讓 `<Picture N>` 對應參照圖上傳順序。兩種模式都會印出**完整的 `PROMPT`**，讓你提交前確認內容。`SHOTS` 的時間需嚴格遞增，且每個時間都小於 `DURATION_SECONDS`；每個描述都必須有實際內容。


In [ ]:
import re

def shot_timestamp(seconds):
    milliseconds = round(seconds * 1000)
    minutes, remainder = divmod(milliseconds, 60_000)
    secs, millis = divmod(remainder, 1_000)
    return f"{minutes:02d}:{secs:02d}.{millis:03d}"

def _format_h3_prompt(p_in, mode_in):
    p_out = str(p_in or "").strip()
    if mode_in in ("t2v", "text_to_video"):
        p_out = re.sub(r"For the target video,\s*at 0\.00 seconds into the target video,[^\n]*\n*", "", p_out, flags=re.IGNORECASE)
        p_out = re.sub(r"\b(in|from|of)\s*<Picture\s+\d+>", "", p_out, flags=re.IGNORECASE)
        p_out = re.sub(r"<Picture\s+\d+>", "", p_out, flags=re.IGNORECASE)
        if "subject_definitions:" in p_out or "[reference generation]" in p_out:
            p_out = re.sub(r"subject_definitions:\s*.*?(?=\n\s*(?:summary:|retention_analysis:|detailed_description:|\[Shot\s*1\]))", "", p_out, flags=re.IGNORECASE | re.DOTALL)
            p_out = re.sub(r"retention_analysis:\s*.*?(?=\n\s*(?:detailed_description:|\[Shot\s*1\]))", "", p_out, flags=re.IGNORECASE | re.DOTALL)
            p_out = re.sub(r"\[reference generation\]\s*", "", p_out, flags=re.IGNORECASE)
            p_out = re.sub(r"summary:\s*.*?(?=\n\s*(?:detailed_description:|\[Shot\s*1\]))", "", p_out, flags=re.IGNORECASE | re.DOTALL)
            p_out = re.sub(r"detailed_description:\s*", "", p_out, flags=re.IGNORECASE)
            p_out = re.sub(r"<Subject\s+1>", "the main character", p_out, flags=re.IGNORECASE)
            p_out = re.sub(r"<Subject\s+\d+>", "the character", p_out, flags=re.IGNORECASE)
        if "integrated_multimodal_description:" not in p_out:
            _raw_p = p_out.strip()
            _snd = "Natural environmental ambience and crisp physical motion sound effects synchronized with the on-screen action."
            _mus = "N/A"
            _m_aud = re.search(r"(?:Synchronized audio|Audio|Soundscape)\s*:\s*(.+)$", _raw_p, flags=re.IGNORECASE | re.DOTALL)
            if _m_aud:
                _snd = _m_aud.group(1).strip()
                _raw_p = _raw_p[:_m_aud.start()].strip()
            _shot_idx = [1]
            def _repl_ts(m):
                s_val = float(m.group(1))
                idx = _shot_idx[0]
                _shot_idx[0] += 1
                if idx == 1 and s_val == 0.0:
                    return "[Shot 1] "
                return f"[Shot {idx}] At {shot_timestamp(s_val)}, the camera cuts to "
            _raw_p = re.sub(r"\bAt\s+(\d+(?:\.\d+)?)\s*s(?:econds?)?\s*,\s*", _repl_ts, _raw_p, flags=re.IGNORECASE)
            if "[Shot 1]" not in _raw_p:
                _raw_p = "[Shot 1] " + _raw_p
            if "overall_soundscape:" in _raw_p:
                p_out = f"integrated_multimodal_description: {_raw_p}"
            else:
                p_out = (
                    f"integrated_multimodal_description: {_raw_p}\n"
                    f"overall_soundscape: {_snd}\n"
                    f"non_diegetic_music: {_mus}"
                )
    elif mode_in == "face_refine":
        if "subject_definitions:" not in p_out:
            _user_d = p_out or "Photorealistic close-up face with natural skin texture, sharp eyes and clear facial details."
            p_out = (
                "subject_definitions:\n"
                "<Subject 1> is the person shown in <Picture 1>; their exact facial identity, eye shape, nose, mouth, skin tone, eyebrows and hair strictly follow <Picture 1>.\n\n"
                "summary:\n"
                "[reference generation] High-detail close-up facial refinement of <Subject 1> from <Picture 1> matching the head pose, lighting and expression timing.\n\n"
                "retention_analysis:\n"
                "<Subject 1> (all shots): fully_preserved - exact facial identity, skin texture and features from <Picture 1>.\n\n"
                "detailed_description:\n"
                f"[Shot 1] Close-up portrait of <Subject 1> from <Picture 1>. {_user_d}\n\n"
                "overall_soundscape: Quiet natural room ambience.\n"
                "non_diegetic_music: N/A"
            )
    elif mode_in in ("char_swap", "v2v"):
        _ed_mode = globals().get("SWAP_EDIT_MODE", "person")
        _ed_tgt = globals().get("SWAP_TARGET", "person") or "person"
        _no_lip = bool(globals().get("SWAP_NO_LIPSYNC", False))
        _lip_line = (
            "Nobody on screen sings, speaks or lip-syncs; the music in <Audio 1> is a background track added later. Mouths stay closed or keep the source expressions such as smiles; the soundtrack itself is unchanged. "
            if _no_lip else
            "The visible people lip-sync any vocals in <Audio 1> where the source people sing or speak; the soundtrack itself is unchanged. "
        )
        if globals().get("USE_MASKED_SWAP", False) and "subject_definitions:" not in p_out:
            if _ed_mode == "remove":
                _bg_fill = p_out or "clean natural background matching the surrounding environment"
                p_out = (
                    "subject_definitions:\n"
                    f"The source footage is <Video 1>, where the {_ed_tgt} is being removed and replaced with {_bg_fill}.\n\n"
                    "summary:\n"
                    f"[video editing + audio reuse] Remove the {_ed_tgt} from <Video 1> so the background behind it shows naturally. Everything else keeps its source look, timing and layout.\n\n"
                    "retention_analysis:\n"
                    f"The source footage (edited source): partially_preserved - the only change is removing the {_ed_tgt}; the people, actions, camera movement, shot composition, cuts and timing are retained.\n"
                    "The soundtrack: fully_copy - the original synchronized soundtrack is the complete final audio.\n\n"
                    "detailed_description:\n"
                    f"[Shot 1] Live-action footage with the source lighting and lens perspective. The area where the {_ed_tgt} was is filled seamlessly with {_bg_fill}. Continue the original camera movement and performance through the last frame.\n\n"
                    "overall_soundscape: Any ambience and physical sounds already present in the source soundtrack remain unchanged.\n"
                    "non_diegetic_music: Reuse the original music and vocals at their original tempo and synchronization."
                )
            elif _ed_mode == "background":
                _bg_desc = p_out or "the environment, lighting and atmosphere shown in <Picture 1>"
                p_out = (
                    "subject_definitions:\n"
                    "<Subject 1> is the environment, props and lighting from <Picture 1>.\n"
                    f"The {_ed_tgt} in <Video 1> is kept unchanged in the foreground.\n\n"
                    "summary:\n"
                    f"[video editing + reference generation + audio reuse] Replace the surroundings around the {_ed_tgt} in <Video 1> with <Subject 1> from <Picture 1> while keeping the {_ed_tgt}, camera movement and soundtrack unchanged.\n\n"
                    "retention_analysis:\n"
                    "<Subject 1> (all shots): fully_preserved - its environment, colours, materials and lighting stay consistent in every frame.\n"
                    f"The source footage (edited source): partially_preserved - the {_ed_tgt} in the foreground, actions, camera movement, cuts and timing are retained.\n\n"
                    "detailed_description:\n"
                    f"[Shot 1] Live-action footage where the {_ed_tgt} from <Video 1> performs in front of <Subject 1> from <Picture 1> ({_bg_desc}). Continue the original camera movement and performance through the last frame.\n\n"
                    "overall_soundscape: Any ambience and physical sounds already present in the source soundtrack remain unchanged.\n"
                    "non_diegetic_music: Reuse the original music and vocals at their original tempo and synchronization."
                )
            elif _ed_mode == "change":
                _chg_desc = p_out or f"the {_ed_tgt} shown in <Picture 1>"
                p_out = (
                    "subject_definitions:\n"
                    f"<Subject 1> is the replacement {_ed_tgt} from <Picture 1> ({_chg_desc}).\n"
                    f"<Video 1> is the source footage being edited, in which the {_ed_tgt} to replace is shown as a smooth grey shape.\n\n"
                    "summary:\n"
                    f"[video editing + reference generation + audio reuse] Replace the {_ed_tgt} in <Video 1> with <Subject 1> from <Picture 1>. Everything else keeps its source look, face, hands, background, timing and layout.\n\n"
                    "retention_analysis:\n"
                    "<Subject 1> (all shots): fully_preserved - its shape, colours, materials and details stay consistent in every frame.\n"
                    f"The source footage (edited source): partially_preserved - the only change is replacing the {_ed_tgt} with <Subject 1>; the person's face, actions, camera movement, shot composition, cuts and timing are retained.\n"
                    "The soundtrack: fully_copy - the original synchronized soundtrack is the complete final audio.\n\n"
                    "detailed_description:\n"
                    f"[Shot 1] Live-action footage with the source lighting and lens perspective. Throughout all shots, <Subject 1> from <Picture 1> ({_chg_desc}) replaces the {_ed_tgt} naturally with the same size, materials and motion in every frame. Continue the original camera movement and performance through the last frame.\n\n"
                    "overall_soundscape: Any ambience and physical sounds already present in the source soundtrack remain unchanged.\n"
                    "non_diegetic_music: Reuse the original music and vocals at their original tempo and synchronization."
                )
            else:
                _usr_extra = (" " + p_out) if p_out and "Replace the person in <Video 1>" not in p_out else ""
                p_out = (
                    "subject_definitions:\n"
                    f"<Subject 1> is the person from <Picture 1>.{_usr_extra} They replace the main person in <Video 1>, in every shot where that person appears. Their face, hair, body and clothing all come from <Picture 1>.\n"
                    "<Subject 2> is the setting, props and lighting in <Video 1>.\n"
                    "<Video 1> is the source footage being edited, in which the person to replace is shown as a smooth grey figure. It supplies the exact camera framing, camera movement, performance timing, body poses, interactions and hard cuts. The grey figure supplies movement only; face, hair, skin and clothing come from <Picture 1>.\n"
                    "<Audio 1> is the synchronized original soundtrack supplied with <Video 1>, reused unchanged as the complete final soundtrack.\n\n"
                    "summary:\n"
                    "[video editing + reference generation + audio reuse] The target video is an edited version of <Video 1>. <Subject 1> takes the place of the main person. Identity and complete outfit come from <Picture 1>. The source performance, camera, cuts, setting, props and soundtrack retain their timing and layout.\n\n"
                    "retention_analysis:\n"
                    "<Subject 1> (all shots): fully_preserved - the referenced facial identity, hair, skin tone and complete outfit remain consistent while poses and mouth movements follow the main person.\n"
                    "<Subject 2> (setting and props): fully_preserved - the location, props and lighting retain their source appearance and position.\n"
                    "<Video 1> (edited source): partially_preserved - the replaced person's identity and outfit change; camera movement, shot composition, actions, interaction, cuts and timing are retained.\n"
                    "<Audio 1>: fully_copy - the original synchronized soundtrack is the complete final audio, without additional generated vocals or sound effects.\n\n"
                    "detailed_description:\n"
                    "[Shot 1] Live-action footage in the same setting as <Video 1>, with natural skin tones and the source lighting and lens perspective. The replacement person appears only where the person they replace is visible; the background of <Picture 1> does not enter the scene. "
                    "<Subject 1> performs every action of the main person: the same gestures, steps, gaze, expressions and mouth shapes at the same moments. "
                    "Throughout all shots, the main person is always the person in <Picture 1>, regardless of screen position or size. Their outfit replaces the original clothing completely; no face, hair, skin, costume, fabric pattern or accessory of the original person remains. "
                    f"{_lip_line}Continue the original camera movement and performance through the last frame without adding transitions or changing speed.\n\n"
                    "overall_soundscape:\n"
                    "Any ambience and physical sounds already present in <Audio 1> remain part of its unchanged mix. Add no new room noise, footsteps, voices or impacts.\n\n"
                    "non_diegetic_music:\n"
                    "Reuse the original music and vocals in <Audio 1> at their original tempo, pitch, mix and synchronization. Do not generate a new performance or additional score."
                )
        elif "<Video 1>" not in p_out:
            p_out = "Replace the person in <Video 1> with the character shown in <Picture 1>. Keep the background, lighting, camera movement, framing, speech and timing of <Video 1> unchanged.\n\n" + p_out
    elif mode_in in ("reference", "r2v") and "subject_definitions:" not in p_out and "integrated_multimodal_description:" not in p_out and "<Video 1>" not in p_out and not p_out.startswith("{"):
        _raw_p = p_out.strip()
        if "<Picture 1>" not in _raw_p:
            _raw_p = "<Picture 1> " + _raw_p
        _snd = "Natural environmental ambience and crisp physical motion sound effects synchronized with the on-screen action."
        _m_aud = re.search(r"(?:Synchronized audio|Audio|Soundscape|overall_soundscape)\s*:\s*(.+)$", _raw_p, flags=re.IGNORECASE | re.DOTALL)
        if _m_aud:
            _snd = _m_aud.group(1).strip()
            _raw_p = _raw_p[:_m_aud.start()].strip()
        if "[Shot 1]" not in _raw_p:
            _raw_p = "[Shot 1] " + _raw_p
        _pics = sorted({int(n) for n in re.findall(r"<Picture\s+(\d+)>", _raw_p) if int(n) >= 1}) or [1]
        _subj_lines = ["<Subject 1> is the main character in <Picture 1>; their facial features, hairstyle, body proportions, and clothing strictly follow <Picture 1>."]
        _ret_lines = ["<Subject 1> is fully preserved from <Picture 1>."]
        for _pk in _pics:
            if _pk > 1:
                _subj_lines.append(f"<Picture {_pk}> provides additional visual/scene reference.")
                _ret_lines.append(f"<Picture {_pk}> visual elements are preserved.")
        p_out = (
            "subject_definitions:\n" + "\n".join(_subj_lines) + "\n\n"
            "summary:\n[reference generation] Photorealistic cinematic sequence of <Subject 1> from <Picture 1> with natural motion and high visual fidelity.\n\n"
            "retention_analysis:\n" + "\n".join(_ret_lines) + "\n\n"
            f"detailed_description:\n{_raw_p}\n\n"
            f"overall_soundscape: {_snd}\n"
            "non_diegetic_music: N/A"
        )
    return p_out

assert isinstance(PROMPT_OVERRIDE, str), "PROMPT_OVERRIDE 必須是字串。"
prompt_file_env = os.environ.get("H3_PROMPT_FILE")
if prompt_file_env:
    prompt_path = Path(prompt_file_env)
    assert prompt_path.is_file(), f"Colab CLI 上傳的 prompt 檔不存在：{prompt_path}"
    PROMPT = prompt_path.read_text(encoding="utf-8")
    assert PROMPT.strip(), f"prompt 檔是空的：{prompt_path}"
    print("使用 CLI 上傳的完整 prompt：", prompt_path)
elif PROMPT_OVERRIDE.strip():
    PROMPT = PROMPT_OVERRIDE.strip()
    print("使用完整自訂 prompt；SHOTS、環境聲與配樂設定不參與組裝。")
elif INFERENCE_MODE == "reference":
    raise ValueError("Ref2VA 需要完整參照格式 prompt；請設定 PROMPT_OVERRIDE 或使用 CLI --prompt 檔案。")
else:
    PROMPT = "Photorealistic cinematic scene, high detail, 8k resolution."

PROMPT = _format_h3_prompt(PROMPT, INFERENCE_MODE)
print("\n模型收到的完整 prompt：\n")
print(PROMPT)


### 3.2 設定圖片參照

FL2VA 模式請提供一張首幀圖片；Ref2VA 模式可提供 1–9 張 PNG、JPEG、WebP 等圖片。Ref2VA 會依輸入順序將它們標記為 `<Picture 1>`、`<Picture 2>` 等，供完整 prompt 描述如何使用。語意參照圖片不會自動固定在影片的不同時間點。

Colab 網頁模式會在下一個 cell 顯示上傳按鈕。本機 CLI 會把每個 `--image` 路徑上傳到遠端，並以 `H3_REFERENCE_IMAGES` 傳入 notebook；CLI 也會上傳 `--prompt` 文字檔，cell 會直接讀取其 UTF-8 內容。


In [ ]:
import json
from io import BytesIO
from PIL import Image

CLI_MODE = False
REFERENCE_IMAGE_NAMES = []
REFERENCE_IMAGE_PATHS = []
SOURCE_VIDEO_NAME = None
LONGSHOT_SONG_NAME = None
REF_AUDIO_NAME = None

if INFERENCE_MODE == "ai_upscale":
    CLI_MODE = True
    input_dir = COMFY / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    src_vid_env = os.environ.get("H3_SOURCE_VIDEO", "").strip()
    assert src_vid_env and Path(src_vid_env).is_file(), f"AI Upscale yêu cầu video nguồn: {src_vid_env}"
    up_in_path = input_dir / "h3_ai_upscale_input.mp4"
    shutil.copy2(src_vid_env, up_in_path)
    SOURCE_VIDEO_NAME = up_in_path.name
    print("模式：AI Latent Upscale x2 (H3VideoUpsampler) — 來源影片：", up_in_path)
elif INFERENCE_MODE in ("t2v", "text_to_video"):
    CLI_MODE = True
    input_dir = COMFY / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    REFERENCE_IMAGE_NAMES = []
    REFERENCE_IMAGE_PATHS = []
    print("模式：Text-to-Video (T2V) — 無需圖片參照，直接由文字提示詞生成高畫質影片。")
elif INFERENCE_MODE in ("reference", "r2v", "director", "char_swap", "v2v", "longshot", "continuum", "face_refine"):
    configured_references = os.environ.get("H3_REFERENCE_IMAGES")
    if configured_references:
        source_paths = json.loads(configured_references)
        if not source_paths and (INFERENCE_MODE in ("director", "t2v", "longshot", "continuum") or os.environ.get("H3_SOURCE_VIDEO", "").strip()):
            source_paths = []
        else:
            assert isinstance(source_paths, list) and 1 <= len(source_paths) <= 9, "Ref2VA 請提供 1–9 張參照圖。"
        source_images = []
        for source_path in source_paths:
            path = Path(source_path)
            assert path.is_file(), f"參照圖片不存在：{path}"
            source_images.append((path.name, path))
        CLI_MODE = True
    else:
        from google.colab import files
        uploaded = files.upload()
        assert 1 <= len(uploaded) <= 9, "Ref2VA 請上傳 1–9 張參照圖片。"
        source_images = [(name, BytesIO(image_bytes)) for name, image_bytes in uploaded.items()]

    input_dir = COMFY / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    if not source_images:
        blank_path = input_dir / "h3_blank_ref.png"
        Image.new("RGB", (64, 64), (0, 0, 0)).save(blank_path)
        REFERENCE_IMAGE_NAMES = [blank_path.name]
        REFERENCE_IMAGE_PATHS = [blank_path]
    else:
        for number, (source_name, image_source) in enumerate(source_images, start=1):
            with Image.open(image_source) as opened:
                image = opened.convert("RGB")
            image_path = input_dir / f"h3_reference_{number:02d}.png"
            image.save(image_path)
            REFERENCE_IMAGE_NAMES.append(image_path.name)
            REFERENCE_IMAGE_PATHS.append(image_path)
            print(f"<Picture {number}>：{source_name} → {image_path} ({image.size[0]}×{image.size[1]})")

    if INFERENCE_MODE in ("char_swap", "v2v", "face_refine") or (INFERENCE_MODE in ("reference", "r2v") and os.environ.get("H3_SOURCE_VIDEO", "").strip()):
        src_vid_env = os.environ.get("H3_SOURCE_VIDEO", "").strip()
        assert src_vid_env and Path(src_vid_env).is_file(), f"Character Swap yêu cầu video nguồn (<Video 1>): {src_vid_env}"
        norm_vid_path = input_dir / "h3_source_video_24fps.mp4"
        probe_a = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=codec_type", "-of", "csv=p=0", src_vid_env],
            capture_output=True, text=True, check=False
        )
        has_audio_stream = bool(probe_a.stdout.strip())
        _sv_w = max(320, (int(WIDTH * 0.56) // 32) * 32) if INFERENCE_MODE in ("reference", "r2v") else WIDTH
        _sv_h = max(320, (int(HEIGHT * 0.56) // 32) * 32) if INFERENCE_MODE in ("reference", "r2v") else HEIGHT
        vf_chain = f"fps=24,scale={_sv_w}:{_sv_h}:force_original_aspect_ratio=decrease,pad={_sv_w}:{_sv_h}:(ow-iw)/2:(oh-ih)/2"
        trim_args = ["-t", f"{DURATION_SECONDS:.3f}"]
        if has_audio_stream:
            subprocess.run([
                "ffmpeg", "-y", "-i", src_vid_env, *trim_args,
                "-vf", vf_chain, "-c:v", "libx264", "-preset", "fast", "-crf", "18",
                "-c:a", "aac", "-b:a", "192k", str(norm_vid_path)
            ], check=True)
        else:
            subprocess.run([
                "ffmpeg", "-y", "-i", src_vid_env, "-f", "lavfi", "-i", "anullsrc=channel_layout=stereo:sample_rate=44100",
                *trim_args, "-vf", vf_chain, "-c:v", "libx264", "-preset", "fast", "-crf", "18",
                "-c:a", "aac", "-shortest", str(norm_vid_path)
            ], check=True)
        SOURCE_VIDEO_NAME = norm_vid_path.name
        print(f"<Video 1> 已標準化為 24fps ({WIDTH}x{HEIGHT}, max {DURATION_SECONDS}s)：{norm_vid_path}")
    if INFERENCE_MODE in ("longshot", "continuum"):
        aud_env = os.environ.get("H3_AUDIO_PATH", "").strip()
        if aud_env and Path(aud_env).is_file():
            song_dest = input_dir / ("h3_longshot_song" + (Path(aud_env).suffix or ".mp3"))
            shutil.copy2(aud_env, song_dest)
            LONGSHOT_SONG_NAME = song_dest.name
            print(f"Long Shot Song Track 音訊已載入：{song_dest}")
    if INFERENCE_MODE in ("reference", "r2v", "char_swap", "v2v") and os.environ.get("H3_AUDIO_AS_REF", "0") == "1":
        aud_ref_env = os.environ.get("H3_AUDIO_PATH", "").strip()
        if aud_ref_env and Path(aud_ref_env).is_file():
            ref_aud_path = input_dir / "h3_ref_audio.wav"
            try:
                subprocess.run(["ffmpeg", "-y", "-i", aud_ref_env, "-t", f"{DURATION_SECONDS:.3f}", "-vn",
                                "-ac", "2", "-ar", "48000", str(ref_aud_path)], check=True, capture_output=True)
                _pr = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                                      str(ref_aud_path)], capture_output=True, text=True, check=False)
                _adur = float((_pr.stdout or "0").strip() or 0)
                if _adur >= 1.0:
                    REF_AUDIO_NAME = ref_aud_path.name
                    print(f"<Audio 1> (lip-sync reference) loaded: {ref_aud_path} ({_adur:.2f}s)")
                else:
                    print(f"<Audio 1> too short ({_adur:.2f}s) -> skip audio conditioning")
            except Exception as _ae:
                print("<Audio 1> conversion failed, skip audio conditioning:", _ae)
else:
    last_frame_path = None
    ref_list = []
    if os.environ.get("H3_REFERENCE_IMAGES"):
        try:
            ref_list = json.loads(os.environ["H3_REFERENCE_IMAGES"])
        except Exception:
            ref_list = []
    configured_image = os.environ.get("H3_INPUT_IMAGE", "").strip()
    if not configured_image and ref_list:
        configured_image = str(ref_list[0])
    source_path = Path(configured_image or "/content/h3_first_frame.png")
    CLI_MODE = source_path.is_file()
    assert not configured_image or CLI_MODE, f"Colab CLI 上傳的首幀不存在：{source_path}"
    if CLI_MODE:
        with Image.open(source_path) as opened:
            image = opened.convert("RGB")
        print("使用 CLI 上傳的首幀：", source_path)
    else:
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) == 1, "FL2VA 請只上傳一張首幀圖片。"
        name, image_bytes = next(iter(uploaded.items()))
        with Image.open(BytesIO(image_bytes)) as opened:
            image = opened.convert("RGB")
        print("使用網頁上傳的首幀：", name)

    if abs(image.width / image.height - WIDTH / HEIGHT) > 0.01:
        print("提醒：原圖比例與輸出不同，首幀會被拉伸到影片畫布。")
    input_path = COMFY / "input" / "h3_colab_first_frame.png"
    input_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(input_path)
    print("ComfyUI 首幀 (Keyframe Đầu)：", input_path, image.size)
    if len(ref_list) >= 2 and Path(str(ref_list[1])).is_file():
        with Image.open(Path(str(ref_list[1]))) as opened_last:
            image_last = opened_last.convert("RGB")
        last_frame_path = COMFY / "input" / "h3_colab_last_frame.png"
        image_last.save(last_frame_path)
        print("ComfyUI 尾幀 (Keyframe Cuối)：", last_frame_path, image_last.size)


## 4. 啟動 ComfyUI 並提交推論

ComfyUI 只在 Colab 的 `127.0.0.1:8188` 執行。第一個 cell 啟動服務並依模式確認節點；第二個 cell 把設定、圖片參照與 `PROMPT` 送入 ComfyUI。若模型或節點名稱不符，會顯示具體錯誤。每次修改第 3 節設定後，請重新執行**設定 → prompt 組裝 → 提交 → 等待輸出**；更換圖片時也要重新執行圖片參照 cell。


In [ ]:
import json, time, requests

PORT = 8188
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/comfyui_h3.log")
try:
    response = requests.get(BASE_URL + "/system_stats", timeout=2)
    response.raise_for_status()
    print("使用已啟動的 ComfyUI")
except requests.RequestException:
    LOG_FILE = open(LOG_PATH, "a")
    launch = [sys.executable, str(COMFY / "main.py"), "--listen", "127.0.0.1", "--port", str(PORT), "--disable-auto-launch"]
    _vram_bytes = torch.cuda.get_device_properties(0).total_memory
    if LOW_VRAM_MODE == "on" or (LOW_VRAM_MODE == "auto" and _vram_bytes < 48 * 2**30):
        launch.append("--lowvram")
    elif _vram_bytes >= 70 * 2**30:
        launch.append("--highvram")
        print("🚀 [A100 80GB MAX-VRAM] Đã bật --highvram --fast: Ghim 100% Model (Qwen3-VL-32B + H3 UNET + Upscaler) trực tiếp trên 80GB VRAM, không offload qua CPU!")
    COMFY_PROCESS = subprocess.Popen(launch, cwd=COMFY, stdout=LOG_FILE, stderr=subprocess.STDOUT)
    for _ in range(180):
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError(f"ComfyUI 啟動失敗；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-5000:])
        try:
            requests.get(BASE_URL + "/system_stats", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            time.sleep(2)
    else:
        raise TimeoutError(f"ComfyUI 啟動逾時；查看 {LOG_PATH}")

object_info = requests.get(BASE_URL + "/object_info", timeout=30).json()
if (INFERENCE_MODE == "longshot" and "MiniMaxH3LongShot" not in object_info) or (INFERENCE_MODE == "continuum" and "H3ContinuumSamplerV39" not in object_info) or (AI_UPSCALE and "H3VideoUpsampler" not in object_info) or ("H3SigmaRefiner" not in object_info and H3_YCNODES.is_dir()) or ("SelfLiftH3Sampler" not in object_info and globals().get("H3_SELFLIFT_NODE") and H3_SELFLIFT_NODE.is_dir()) or ("MiniMaxChunkFeedForward" not in object_info and globals().get("H3_KJNODES") and H3_KJNODES.is_dir()) or ("FrameInterpolationModelLoader" not in object_info and (COMFY / "comfy_extras" / "nodes_frame_interp.py").is_file()) or ("ImageDetailContrast" not in object_info and globals().get("H3_DETAIL_CONTRAST") and H3_DETAIL_CONTRAST.is_dir()) or ("Orion4D_FXMax_DCTLToneMapper" not in object_info and globals().get("H3_ORION4D") and H3_ORION4D.is_dir()) or ("H3PeopleMask" not in object_info and globals().get("H3_ONECLICK_SWAP") and H3_ONECLICK_SWAP.is_dir()) or ("MiniMaxH3FantasticVideoEditLatent" not in object_info and globals().get("H3_FANTASTIC") and H3_FANTASTIC.is_dir()) or ("MiniMaxH3ContexMaskedTarget" not in object_info and globals().get("H3_CONTEXT_LOOP") and H3_CONTEXT_LOOP.is_dir()) or ("H3FaceTrackCrop" not in object_info and globals().get("H3_FACEREFINE") and H3_FACEREFINE.is_dir()):
    print("偵測到新安裝的 H3_Longshot 節點，正在重啟 ComfyUI 以載入節點…")
    subprocess.run(["pkill", "-f", "ComfyUI/main.py"], check=False)
    time.sleep(2)
    LOG_FILE = open(LOG_PATH, "a")
    launch = [sys.executable, str(COMFY / "main.py"), "--listen", "127.0.0.1", "--port", str(PORT), "--disable-auto-launch"]
    _vram_bytes = torch.cuda.get_device_properties(0).total_memory
    if LOW_VRAM_MODE == "on" or (LOW_VRAM_MODE == "auto" and _vram_bytes < 48 * 2**30):
        launch.append("--lowvram")
    elif _vram_bytes >= 70 * 2**30:
        launch.append("--highvram")
        print("🚀 [A100 80GB MAX-VRAM] Đã bật --highvram --fast: Ghim 100% Model (Qwen3-VL-32B + H3 UNET + Upscaler) trực tiếp trên 80GB VRAM, không offload qua CPU!")
    COMFY_PROCESS = subprocess.Popen(launch, cwd=COMFY, stdout=LOG_FILE, stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            requests.get(BASE_URL + "/system_stats", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            time.sleep(2)
    object_info = requests.get(BASE_URL + "/object_info", timeout=30).json()
required_nodes = {"UNETLoader", "CLIPLoader", "VAELoader", "LoraLoaderModelOnly",
                  "LoadImage", "RandomNoise", "KSamplerSelect",
                  "BasicScheduler", "VAEDecode", "VAEDecodeAudio",
                  "VHS_VideoCombine"}
if INFERENCE_MODE == "longshot":
    required_nodes.update({"MiniMaxH3TimelineShot", "MiniMaxH3RefPromptBuilder", "MiniMaxH3Subject", "MiniMaxH3LongShot"})
elif INFERENCE_MODE == "ai_upscale":
    required_nodes = {"UNETLoader", "CLIPLoader", "VAELoader", "LoadVideo", "GetVideoComponents", "H3VideoUpsampler", "VHS_VideoCombine"}
elif INFERENCE_MODE == "continuum":
    required_nodes.update({"H3ContinuumReferenceImagesV39", "H3ContinuumSamplerV39", "H3DecodeCacheHelper", "H3ContinuumAssembleSeamV35"})
elif INFERENCE_MODE == "face_refine":
    required_nodes.update({"LoadVideo", "GetVideoComponents", "H3FaceTrackCrop", "H3InjectVideoLatent", "H3FaceStitch", "MiniMaxH3ReferenceToVideo", "BasicGuider", "SamplerCustomAdvanced"})
elif INFERENCE_MODE in ("reference", "r2v", "t2v", "text_to_video", "director", "char_swap", "v2v"):
    required_nodes.update({"BasicGuider", "SamplerCustomAdvanced", "MiniMaxH3ReferenceToVideo"})
    if INFERENCE_MODE in ("char_swap", "v2v") or (INFERENCE_MODE in ("reference", "r2v") and os.environ.get("H3_SOURCE_VIDEO", "").strip()):
        required_nodes.update({"LoadVideo", "GetVideoComponents"})
else:
    required_nodes.update({"ImageScaleToTotalPixels", "MiniMaxH3ImageToVideo"})
if globals().get("REF_AUDIO_NAME"):
    required_nodes.add("LoadAudio")
if AI_UPSCALE:
    required_nodes.add("H3VideoUpsampler")
missing = required_nodes - object_info.keys()
assert not missing, f"ComfyUI 缺少節點：{sorted(missing)}。請查看 {LOG_PATH}，並確認 ComfyUI / VideoHelperSuite 安裝成功。"
print("節點檢查通過，ComfyUI 已啟動")


In [ ]:
import re

graph = {
    "1": {"class_type": "UNETLoader", "inputs": {"unet_name": DIFFUSION, "weight_dtype": "default"}},
    "3": {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENCODER, "type": "minimax", "device": "default"}},
    "4": {"class_type": "VAELoader", "inputs": {"vae_name": VIDEO_VAE}},
    "5": {"class_type": "VAELoader", "inputs": {"vae_name": AUDIO_VAE}},
}

base_model_node = "1"
if "MiniMaxH3SigmaShift" in object_info:
    _shift_v = 10.0 if LORA_CHOICE in ("fasth3_t2v", "fasth3_trim") else 12.0
    graph["1_shift"] = {"class_type": "MiniMaxH3SigmaShift", "inputs": {"model": ["1", 0], "shift_video": _shift_v, "shift_audio": 3.0}}
    base_model_node = "1_shift"

if "ModelAttentionBackend" in object_info:
    _ab_opts = object_info.get("ModelAttentionBackend", {}).get("input", {}).get("required", {}).get("attention", [[]])[0]
    _ab_val = "comfy kitchen attention" if (isinstance(_ab_opts, list) and "comfy kitchen attention" in _ab_opts) else "pytorch attention"
    graph["1_attn"] = {"class_type": "ModelAttentionBackend", "inputs": {"model": [base_model_node, 0], "attention": _ab_val}}
    base_model_node = "1_attn"

if "MiniMaxLowVRAMAttention" in object_info and os.environ.get("H3_LOWVRAM_ATTN", "1") != "0":
    # head_chunks=1 khi bật SelfLiftH3TST trên GPU >=22GB để giữ chuẩn bộ đếm layer của TST nhưng vẫn giải phóng sớm bộ đệm QKV
    _use_tst = ("SelfLiftH3TST" in object_info and os.environ.get("H3_TST", "1") != "0")
    _head_chunks = 1 if (_use_tst and VRAM_GIB >= 22.0) else 2
    graph["1_lowvram_attn"] = {"class_type": "MiniMaxLowVRAMAttention", "inputs": {"model": [base_model_node, 0], "head_chunks": _head_chunks}}
    base_model_node = "1_lowvram_attn"
    print(f"🛡️ [KJNodes] Đã bật MiniMaxLowVRAMAttention (head_chunks={_head_chunks}) — Giải phóng tức thì QKV buffer!")

if "MiniMaxChunkFeedForward" in object_info and os.environ.get("H3_CHUNK_FFN", "1") != "0":
    graph["1_chunk_ffn"] = {"class_type": "MiniMaxChunkFeedForward", "inputs": {"model": [base_model_node, 0], "chunks": 2, "seq_threshold": 4096}}
    base_model_node = "1_chunk_ffn"
    print("🛡️ [KJNodes] Đã bật MiniMaxChunkFeedForward (chunks=2, seq_threshold=4096) — Chống tràn VRAM khi tạo video dài!")
if LORA_CHOICE in ("fasth3_omni", "fasth3_t2v", "fasth3_trim"):
    if "BlockSparseAttention" in object_info:
        graph["1_vsa"] = {
            "class_type": "BlockSparseAttention",
            "inputs": {
                "model": [base_model_node, 0],
                "selection": "vsa",
                "selection.keep_percent": 10.0,
                "start_percent": 0.2,
                "end_percent": 1.0,
                "dense_blocks": "",
                "min_tokens": 12288,
                "extra_tokens": 256,
                "sink_conditioning": "exact_kv_and_rows",
                "verbose": False,
            },
        }
        base_model_node = "1_vsa"

if LORA:
    graph["2"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": [base_model_node, 0], "lora_name": LORA, "strength_model": LORA_STRENGTH}}
    active_model_node = "2"
else:
    active_model_node = base_model_node

if SWAP_LORA:
    graph["2_swap"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": [active_model_node, 0], "lora_name": SWAP_LORA, "strength_model": 0.9}}
    active_model_node = "2_swap"

if globals().get("LMS_SHARP_LORA"):
    _lms_str = float(os.environ.get("H3_LMS_STRENGTH", "0.65" if LORA_CHOICE == "lms_sharp" else "0.45"))
    graph["2_lms"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": [active_model_node, 0], "lora_name": LMS_SHARP_LORA, "strength_model": _lms_str}}
    active_model_node = "2_lms"
    print(f"💎 [Open-H3-Regenerate-2K] Đã kích hoạt LMS Sharpness LoRA ({LMS_SHARP_LORA}, strength={_lms_str}) + 3D Latent Upscaler LMS v0.1!")

if INFERENCE_MODE == "face_refine":
    for index, image_name in enumerate(REFERENCE_IMAGE_NAMES[:1]):
        graph[str(100 + index)] = {"class_type": "LoadImage", "inputs": {"image": image_name}}
    graph.update({
        "200": {"class_type": "LoadVideo", "inputs": {"file": SOURCE_VIDEO_NAME}},
        "201": {"class_type": "GetVideoComponents", "inputs": {"video": ["200", 0]}},
        "9": {"class_type": "RandomNoise", "inputs": {"noise_seed": SEED}},
        "10": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": SAMPLER_NAME}},
        "15": {
            "class_type": "VHS_VideoCombine",
            "inputs": {
                "images": ["201", 0], "audio": ["201", 1], "frame_rate": 24,
                "loop_count": 0, "filename_prefix": OUTPUT_PREFIX,
                "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": VIDEO_CRF,
                "save_metadata": True, "trim_to_audio": False,
                "pingpong": False, "save_output": True,
            },
        },
    })
elif INFERENCE_MODE == "ai_upscale":
    graph.update({
        "200": {"class_type": "LoadVideo", "inputs": {"file": SOURCE_VIDEO_NAME}},
        "201": {"class_type": "GetVideoComponents", "inputs": {"video": ["200", 0]}},
        "14_ai_up": {
            "class_type": "H3VideoUpsampler",
            "inputs": {
                "model": ["1", 0],
                "clip": ["3", 0],
                "vae": ["4", 0],
                "images": ["201", 0],
                "audio": ["201", 1],
                "audio_vae": ["5", 0],
                "prompt": "h3upscale, increase resolution and restore fine details while preserving the original content and visual style.",
                "upscale_model": "minimax_h3_latent_upscaler_3d_bf16.safetensors",
                "megapixels": 2.0,
                "denoise": 0.1,
                "lora": "h3upscale_v2_x2.safetensors",
                "lora_strength": 1.0,
                "tile_tokens": 180000,
                "sampler_name": "euler",
            },
        },
        "15": {
            "class_type": "VHS_VideoCombine",
            "inputs": {
                "images": ["14_ai_up", 0], "audio": ["14_ai_up", 1], "frame_rate": 24,
                "loop_count": 0, "filename_prefix": OUTPUT_PREFIX,
                "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": VIDEO_CRF,
                "save_metadata": True, "trim_to_audio": False,
                "pingpong": False, "save_output": True,
            },
        },
    })
elif INFERENCE_MODE == "longshot":
    ls_data = {}
    if PROMPT.strip().startswith("{"):
        try:
            ls_data = json.loads(PROMPT)
        except Exception:
            ls_data = {}
    ls_shots = ls_data.get("shots") or []
    if not ls_shots:
        parts = [p.strip() for p in re.split(r"(?:^|\n)\s*\[Shot\s*\d+\]", PROMPT) if p.strip()]
        if not parts:
            parts = [PROMPT.strip()]
        per_sec = max(5.0, min(15.0, round(DURATION_SECONDS / max(1, len(parts)), 1)))
        ls_shots = [{"id": f"s{i+1}", "seconds": per_sec, "join": "bridge", "text": pt} for i, pt in enumerate(parts)]
    ls_cast = ls_data.get("cast") or []
    ls_style = ls_data.get("style") or {}
    ls_overlap = int(ls_data.get("overlap", 22))
    if ls_overlap not in {5, 22, 39, 56, 73, 90, 107}:
        ls_overlap = 22

    graph.update({
        "9": {"class_type": "RandomNoise", "inputs": {"noise_seed": SEED}},
        "10": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": SAMPLER_NAME}},
        "11": {"class_type": "BasicScheduler", "inputs": {"model": [active_model_node, 0], "scheduler": SCHEDULER_NAME, "steps": STEPS, "denoise": DENOISE}},
    })
    _ls_sigmas = ["11", 0]

    ls_refs = {}
    subjects_link = None
    for index, image_name in enumerate(REFERENCE_IMAGE_NAMES):
        node_id = str(100 + index)
        graph[node_id] = {"class_type": "LoadImage", "inputs": {"image": image_name}}
        if "ImageScaleToMaxDimension" in object_info:
            res_id = f"ls_resize_{index + 1}"
            graph[res_id] = {"class_type": "ImageScaleToMaxDimension", "inputs": {"image": [node_id, 0], "upscale_method": "lanczos", "largest_size": 1500}}
            ls_refs[f"ref_images.ref_image_{index}"] = [res_id, 0]
        else:
            ls_refs[f"ref_images.ref_image_{index}"] = [node_id, 0]
        c_info = ls_cast[index] if index < len(ls_cast) and isinstance(ls_cast[index], dict) else {}
        pic_tag = f"<Picture {index + 1}>"
        subj_inputs = {
            "label": str(c_info.get("label") or pic_tag),
            "description": str(c_info.get("desc") or f"Character/scene reference in {pic_tag}"),
            "role": str(c_info.get("role") or "appearance"),
            "reference": pic_tag,
            "role_2": "-",
            "reference_2": "",
        }
        if subjects_link is not None:
            subj_inputs["subjects"] = subjects_link
        subj_id = f"ls_subject_{index + 1}"
        graph[subj_id] = {"class_type": "MiniMaxH3Subject", "inputs": subj_inputs}
        subjects_link = [subj_id, 0]

    shots_link = None
    for s_idx, sh in enumerate(ls_shots, start=1):
        s_sec = max(4.0, min(15.0, float(sh.get("seconds", 8.0))))
        j_raw = str(sh.get("join") or "bridge").strip().lower()
        j_val = "cut" if j_raw in ("cut", "context") else "bridge"
        s_text = str(sh.get("text") or "").strip()
        if j_raw == "context":
            s_text = "The shot opens on a new camera angle and composition while preserving the characters, outfit and lighting. " + s_text
        sh_inputs = {
            "shot_id": str(sh.get("id") or f"s{s_idx}"),
            "seconds": s_sec,
            "text": s_text,
            "shot_seed": int(sh.get("shot_seed", -1)),
            "lock": "",
            "join": j_val,
            "frames": 0,
        }
        if shots_link is not None:
            sh_inputs["shots"] = shots_link
        sh_id = f"ls_shot_{s_idx}"
        graph[sh_id] = {"class_type": "MiniMaxH3TimelineShot", "inputs": sh_inputs}
        shots_link = [sh_id, 0]

    builder_inputs = {
        "subject_definitions": "",
        "task_type": "reference generation",
        "task_type_2": "-",
        "task_type_3": "-",
        "summary": str(ls_style.get("summary") or "Continuous multi-shot cinematic video maintaining strict character and visual consistency."),
        "retention_analysis": str(ls_style.get("retention") or "Strictly preserve character facial identity, hairstyle, body proportions and outfit across all shots."),
        "style_line": str(ls_style.get("style_line") or "Live-action photorealistic 8k cinema, natural skin texture, smooth motion."),
        "detailed_description": "",
        "overall_soundscape": str(ls_style.get("soundscape") or OVERALL_SOUNDSCAPE),
        "non_diegetic_music": str(ls_style.get("music") or NON_DIEGETIC_MUSIC),
        "shots": shots_link,
    }
    if subjects_link is not None:
        builder_inputs["subjects"] = subjects_link
    graph["ls_builder"] = {"class_type": "MiniMaxH3RefPromptBuilder", "inputs": builder_inputs}

    if LONGSHOT_SONG_NAME and "LoadAudio" in object_info and "MiniMaxH3SongTrack" in object_info:
        graph["ls_audio"] = {"class_type": "LoadAudio", "inputs": {"audio": LONGSHOT_SONG_NAME}}
        graph["ls_song"] = {"class_type": "MiniMaxH3SongTrack", "inputs": {"audio": ["ls_audio", 0], "audio_vae": ["5", 0]}}
        ls_refs["song"] = ["ls_song", 0]

    graph["12"] = {
        "class_type": "MiniMaxH3LongShot",
        "inputs": {
            "model": [active_model_node, 0],
            "clip": ["3", 0],
            "vae": ["4", 0],
            "noise": ["9", 0],
            "sampler": ["10", 0],
            "sigmas": _ls_sigmas,
            "prompt": ["ls_builder", 3],
            "width": WIDTH,
            "height": HEIGHT,
            "overlap_frames": ls_overlap,
            "seed_mode": "increment",
            "dry_run": False,
            "reuse_segments": False,
            "ref_image_size": REFERENCE_IMAGE_SIZE,
            "save_to_disk": False,
            "cache_name": OUTPUT_PREFIX[:32],
            "audio_vae": ["5", 0],
            **ls_refs,
        },
    }
    graph.update({
        "13": {"class_type": "VAEDecode", "inputs": {"samples": ["12", 0], "vae": ["4", 0]}},
        "14": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["12", 0], "vae": ["5", 0]}},
        "15": {"class_type": "VHS_VideoCombine", "inputs": {
            "images": ["13", 0], "audio": ["14", 0], "frame_rate": 24,
            "loop_count": 0, "filename_prefix": OUTPUT_PREFIX,
            "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": VIDEO_CRF,
            "save_metadata": True, "trim_to_audio": False,
            "pingpong": False, "save_output": True}},
    })
    if (
        globals().get("USE_MASKED_SWAP", False)
        and INFERENCE_MODE in ("char_swap", "v2v")
        and "201" in graph
        and "7" in graph
        and "12" in graph
        and {"CheckpointLoaderSimple", "CLIPTextEncode", "SAM3_VideoTrack", "H3PeopleMask", "H3GreyMannequins", "GrowMaskWithBlur", "MiniMaxH3FantasticVideoEditLatent"} <= object_info.keys()
    ):
        _ed_mode = globals().get("SWAP_EDIT_MODE", "person")
        _ed_tgt = globals().get("SWAP_TARGET", "person") or "person"
        _sam_target = "person" if _ed_mode in ("person", "background") else _ed_tgt
        _is_person = (_ed_mode == "person")
        _hat_fix = bool(globals().get("SWAP_HAT_FIX", False))
        _no_lip = bool(globals().get("SWAP_NO_LIPSYNC", False))

        graph["oj_sam_ckpt"] = {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "sam3.1_multiplex_fp16.safetensors"}}
        graph["oj_sam_txt"] = {"class_type": "CLIPTextEncode", "inputs": {"text": _sam_target, "clip": ["oj_sam_ckpt", 1]}}
        graph["oj_sam_scale"] = {"class_type": "ImageScaleToMaxDimension", "inputs": {"upscale_method": "area", "largest_size": 1008, "image": ["201", 0]}}
        graph["oj_mot_scale"] = {"class_type": "ImageScaleToMaxDimension", "inputs": {"upscale_method": "area", "largest_size": 512, "image": ["201", 0]}}
        graph["oj_sam_track"] = {
            "class_type": "SAM3_VideoTrack",
            "inputs": {
                "detection_threshold": 0.35,
                "max_objects": 0,
                "detect_interval": 1,
                "images": ["oj_sam_scale", 0],
                "model": ["oj_sam_ckpt", 0],
                "conditioning": ["oj_sam_txt", 0],
            },
        }
        graph["oj_mask"] = {
            "class_type": "H3PeopleMask",
            "inputs": {
                "track_data": ["oj_sam_track", 0],
                "headroom": 0.1 if _is_person else 0.0,
                "hold_frames": 2,
                "max_people": 1 if _is_person else 0,
            },
        }
        graph["oj_mask_tight"] = {
            "class_type": "H3PeopleMask",
            "inputs": {
                "track_data": ["oj_sam_track", 0],
                "headroom": 0.0,
                "hold_frames": 1,
                "max_people": 1 if _is_person else 0,
            },
        }
        _mask_for_grow = ["oj_mask", 0]
        if _hat_fix and _is_person and {"SAM3_TrackToMask", "MaskComposite"} <= object_info.keys():
            graph["oj_hat_txt"] = {"class_type": "CLIPTextEncode", "inputs": {"text": "hat", "clip": ["oj_sam_ckpt", 1]}}
            graph["oj_hat_track"] = {
                "class_type": "SAM3_VideoTrack",
                "inputs": {
                    "detection_threshold": 0.35,
                    "max_objects": 0,
                    "detect_interval": 1,
                    "images": ["oj_sam_scale", 0],
                    "model": ["oj_sam_ckpt", 0],
                    "conditioning": ["oj_hat_txt", 0],
                },
            }
            graph["oj_hat_mask"] = {"class_type": "SAM3_TrackToMask", "inputs": {"object_indices": "", "track_data": ["oj_hat_track", 0]}}
            graph["oj_hat_search"] = {
                "class_type": "GrowMaskWithBlur",
                "inputs": {
                    "expand": 60, "incremental_expandrate": 0, "tapered_corners": True,
                    "flip_input": False, "blur_radius": 0, "lerp_alpha": 1.0,
                    "decay_factor": 1.0, "fill_holes": True, "mask": ["oj_mask", 0],
                },
            }
            graph["oj_hat_only"] = {
                "class_type": "MaskComposite",
                "inputs": {"x": 0, "y": 0, "operation": "multiply", "destination": ["oj_hat_mask", 0], "source": ["oj_hat_search", 0]},
            }
            graph["oj_person_plus_hat"] = {
                "class_type": "MaskComposite",
                "inputs": {"x": 0, "y": 0, "operation": "add", "destination": ["oj_mask", 0], "source": ["oj_hat_only", 0]},
            }
            _mask_for_grow = ["oj_person_plus_hat", 0]

        _man_inputs = {
            "blur": 0.04,
            "face_blur": 0.05,
            "images": ["oj_mot_scale", 0],
            "mask": ["oj_mask_tight", 0],
            "size": ["oj_mask_tight", 1],
        }
        if _no_lip and _is_person:
            graph["oj_face_txt"] = {"class_type": "CLIPTextEncode", "inputs": {"text": "face", "clip": ["oj_sam_ckpt", 1]}}
            graph["oj_face_track"] = {
                "class_type": "SAM3_VideoTrack",
                "inputs": {
                    "detection_threshold": 0.2,
                    "max_objects": 0,
                    "detect_interval": 1,
                    "images": ["oj_sam_scale", 0],
                    "model": ["oj_sam_ckpt", 0],
                    "conditioning": ["oj_face_txt", 0],
                },
            }
            graph["oj_face_mask"] = {
                "class_type": "H3PeopleMask",
                "inputs": {"headroom": 0.0, "hold_frames": 0, "max_people": 0, "track_data": ["oj_face_track", 0]},
            }
            _man_inputs["faces"] = ["oj_face_mask", 0]
            _man_inputs["face_size"] = ["oj_face_mask", 1]

        graph["oj_mannequin"] = {"class_type": "H3GreyMannequins", "inputs": _man_inputs}
        graph["oj_grow"] = {
            "class_type": "GrowMaskWithBlur",
            "inputs": {
                "expand": 40 if _is_person else 20,
                "incremental_expandrate": 0,
                "tapered_corners": True,
                "flip_input": (_ed_mode == "background"),
                "blur_radius": 0,
                "lerp_alpha": 1.0,
                "decay_factor": 1.0,
                "fill_holes": True,
                "mask": _mask_for_grow,
            },
        }
        _edit_aud_in = ["201", 1]
        if _no_lip and "AudioAdjustVolume" in object_info:
            graph["oj_mute_aud"] = {"class_type": "AudioAdjustVolume", "inputs": {"volume": -100, "audio": ["201", 1]}}
            _edit_aud_in = ["oj_mute_aud", 0]
            if "ref_video_audios.ref_video_audio_0" in graph["7"]["inputs"]:
                graph["7"]["inputs"]["ref_video_audios.ref_video_audio_0"] = ["oj_mute_aud", 0]
            graph["15"]["inputs"]["audio"] = ["201", 1]

        graph["oj_edit_lat"] = {
            "class_type": "MiniMaxH3FantasticVideoEditLatent",
            "inputs": {
                "grow": 0,
                "keep_audio": True,
                "megapixels": 0.0,
                "vae": ["4", 0],
                "images": ["201", 0],
                "mask": ["oj_grow", 0],
                "audio_vae": ["5", 0],
                "audio": _edit_aud_in,
            },
        }
        graph["7"]["inputs"]["width"] = ["oj_edit_lat", 1]
        graph["7"]["inputs"]["height"] = ["oj_edit_lat", 2]
        graph["7"]["inputs"]["length"] = ["oj_edit_lat", 3]
        if _ed_mode == "remove":
            graph["7"]["inputs"].pop("ref_videos.ref_video_0", None)
            for _ri_k in list(graph["7"]["inputs"].keys()):
                if _ri_k.startswith("ref_images."):
                    graph["7"]["inputs"].pop(_ri_k, None)
        else:
            graph["7"]["inputs"]["ref_videos.ref_video_0"] = ["oj_mannequin", 0]

        if "MiniMaxH3ContexMaskedTarget" in object_info:
            graph["oj_masked_target"] = {
                "class_type": "MiniMaxH3ContexMaskedTarget",
                "inputs": {
                    "mask_meaning": "white = generate",
                    "audio_mode": "preserve source audio",
                    "mask_conversion": "H3 exact (causal/token max)",
                    "target_latent": ["oj_edit_lat", 0],
                    "mask": ["oj_grow", 0],
                },
            }
            graph["12"]["inputs"]["latent_image"] = ["oj_masked_target", 0]
        else:
            graph["12"]["inputs"]["latent_image"] = ["oj_edit_lat", 0]
        print(f"🪄 [OpenJutsu Masked Swap] Đã bật SAM 3.1 + H3GreyMannequins + Masked Target (mode={_ed_mode}, target='{_sam_target}', hat_fix={_hat_fix}, no_lipsync={_no_lip}) — Khóa 100% nền gốc ngoài vùng mask!")
elif INFERENCE_MODE == "continuum":
    cont_data = {}
    if PROMPT.strip().startswith("{"):
        try:
            cont_data = json.loads(PROMPT)
        except Exception:
            cont_data = {}
    seq_prompt_text = str(cont_data.get("sequence_prompt") or cont_data.get("prompt") or "").strip()
    if not seq_prompt_text and isinstance(cont_data.get("chunk_prompts"), list):
        seq_prompt_text = "\n\n".join(
            f"[Chunk {i + 1}]\n{str(p).strip()}" for i, p in enumerate(cont_data["chunk_prompts"]) if str(p).strip()
        )
    if not seq_prompt_text:
        seq_prompt_text = PROMPT.strip()
    # Auto-convert <Picture k> to @Rk for V3.9 reference tag routing compatibility
    seq_prompt_text = re.sub(r"<Picture\s+(\d)>", r"@R\1", seq_prompt_text, flags=re.IGNORECASE)

    chunk_headers = re.findall(r"(?m)^\s*\[Chunk\s+\d+\]", seq_prompt_text, flags=re.IGNORECASE)
    default_chunks = len(chunk_headers) if chunk_headers else max(2, min(16, round(DURATION_SECONDS / 10.0)))
    cont_chunks = max(1, min(16, int(cont_data.get("chunks") or default_chunks)))
    cont_chunk_sec = max(4.0, min(30.0, float(cont_data.get("chunk_seconds") or round(DURATION_SECONDS / max(1, cont_chunks), 1))))
    cont_prompt_mode = str(cont_data.get("prompt_mode") or ("Timeline" if chunk_headers else "Auto"))
    cont_continuity = str(cont_data.get("continuity") or "Balanced")
    cont_backend = str(cont_data.get("continuation_backend") or "Standard")
    cont_video_seam = str(cont_data.get("video_seam") or "Auto")
    cont_audio_seam = str(cont_data.get("audio_seam") or "Auto")
    cont_audio_cont = bool(cont_data.get("audio_continuity", True))
    cont_ref_use = str(cont_data.get("reference_use") or "All chunks")
    cont_r_chunks = cont_data.get("r_chunks") if isinstance(cont_data.get("r_chunks"), dict) else {}

    sampler_req = object_info.get("H3ContinuumSamplerV39", {}).get("input", {}).get("required", {})
    def _pick_sampler_opt(field, preferred, prefix=""):
        opts = sampler_req.get(field, [[]])[0]
        if not isinstance(opts, list) or not opts:
            return preferred
        if preferred in opts:
            return preferred
        for o in opts:
            if str(o).lower().startswith((prefix or preferred).lower()):
                return o
        return opts[0]

    graph.update({
        "10": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": SAMPLER_NAME}},
        "11": {"class_type": "BasicScheduler", "inputs": {"model": [active_model_node, 0], "scheduler": SCHEDULER_NAME, "steps": STEPS, "denoise": DENOISE}},
    })
    _ls_sigmas = ["11", 0]

    ref_v39_inputs = {
        "reference_use": "Per chunk" if cont_ref_use.lower() == "per chunk" else "All chunks",
    }
    for r_i in range(1, 10):
        custom_sel = str(cont_r_chunks.get(f"r{r_i}") or cont_r_chunks.get(str(r_i)) or "").strip()
        if not custom_sel:
            custom_sel = f"1-{cont_chunks}" if r_i <= len(REFERENCE_IMAGE_NAMES) else "off"
        ref_v39_inputs[f"reference_r{r_i}_chunks"] = custom_sel

    for index, image_name in enumerate(REFERENCE_IMAGE_NAMES[:9]):
        node_id = str(100 + index)
        graph[node_id] = {"class_type": "LoadImage", "inputs": {"image": image_name}}
        ref_v39_inputs[f"reference_image_{index + 1}"] = [node_id, 0]
    graph["cont_refs"] = {"class_type": "H3ContinuumReferenceImagesV39", "inputs": ref_v39_inputs}

    if "PrimitiveStringMultiline" in object_info:
        graph["cont_prompt"] = {"class_type": "PrimitiveStringMultiline", "inputs": {"value": seq_prompt_text}}
        seq_prompt_val = ["cont_prompt", 0]
    else:
        seq_prompt_val = seq_prompt_text

    ref_size_label = "Match Output" if REFERENCE_IMAGE_SIZE == "match" else "Max Identity"
    sampler_inputs = {
        "model": [active_model_node, 0],
        "clip": ["3", 0],
        "video_vae": ["4", 0],
        "sampler": ["10", 0],
        "sigmas": ["11", 0],
        "sequence_prompt": seq_prompt_val,
        "prompt_mode": _pick_sampler_opt("prompt_mode", cont_prompt_mode),
        "chunks": cont_chunks,
        "chunk_seconds": cont_chunk_sec,
        "aspect": _pick_sampler_opt("aspect", "Auto from First Image", "Auto"),
        "preset": _pick_sampler_opt("preset", "Draft — 0.30 MP", "Draft"),
        "custom_mp": 0.3,
        "continuity": _pick_sampler_opt("continuity", cont_continuity, cont_continuity.split()[0]),
        "base_seed": SEED,
        "audio_continuity": cont_audio_cont,
        "diagnostics": _pick_sampler_opt("diagnostics", "Detailed Report", "Detailed"),
        "reroll_from_chunk": "Auto",
        "reroll_nonce": 0,
        "strict_compatibility": False,
        "debug": False,
        "show_preview": False,
        "run_storage": "Off",
        "run_name": "",
        "reference_size": _pick_sampler_opt("reference_size", ref_size_label, "Match" if REFERENCE_IMAGE_SIZE == "match" else "Max"),
        "project_id": "",
        "video_reference_size": _pick_sampler_opt("video_reference_size", "Efficient — 0.4 MP", "Efficient"),
        "continuation_backend": _pick_sampler_opt("continuation_backend", cont_backend, cont_backend),
        "generation_mode": _pick_sampler_opt("generation_mode", "Full Run", "Full"),
        "review_action": _pick_sampler_opt("review_action", "Continue / Next", "Continue"),
        "take_group": 0,
        "take_revision_id": "",
        "take_action": _pick_sampler_opt("take_action", "Automatic", "Auto"),
        "size_source": _pick_sampler_opt("size_source", "Manual", "Manual"),
        "width": WIDTH,
        "height": HEIGHT,
        "reference_images": ["cont_refs", 0],
        "audio_vae": ["5", 0],
    }
    if LONGSHOT_SONG_NAME and "LoadAudio" in object_info:
        graph["cont_audio"] = {"class_type": "LoadAudio", "inputs": {"audio": LONGSHOT_SONG_NAME}}
        sampler_inputs["driving_audio"] = ["cont_audio", 0]

    graph["12"] = {"class_type": "H3ContinuumSamplerV39", "inputs": sampler_inputs}
    graph["13_dec"] = {
        "class_type": "H3DecodeCacheHelper",
        "inputs": {
            "cache_mode": "Auto",
            "ram_budget_mb": 256,
            "disk_budget_gb": 8,
            "reset_token": 0,
            "video_samples": ["12", 0],
            "video_vae": ["4", 0],
            "audio_samples": ["12", 1],
            "audio_vae": ["5", 0],
        },
    }
    seam_inputs = {
        "images": ["13_dec", 0],
        "audio": ["13_dec", 1],
        "assembly_plan": ["12", 2],
        "exact_total_duration": True,
        "audio_seam": cont_audio_seam if cont_audio_seam in ("Auto", "Off") else "Auto",
        "video_seam": cont_video_seam if cont_video_seam in ("Auto", "Auto 2", "Analyze Only", "Off") else "Auto",
        "buffer_backend": "Auto",
        "diagnostics": "Detailed Report",
        "driving_audio": ["12", 4],
    }
    graph["14_seam"] = {"class_type": "H3ContinuumAssembleSeamV35", "inputs": seam_inputs}
    graph["15"] = {
        "class_type": "VHS_VideoCombine",
        "inputs": {
            "images": ["14_seam", 0], "audio": ["14_seam", 1], "frame_rate": 24,
            "loop_count": 0, "filename_prefix": OUTPUT_PREFIX,
            "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": VIDEO_CRF,
            "save_metadata": True, "trim_to_audio": False,
            "pingpong": False, "save_output": True,
        },
    }
elif INFERENCE_MODE in ("t2v", "text_to_video") and LORA_CHOICE in ("fasth3_t2v", "fasth3_trim") and "MiniMaxH3ImageToVideo" in object_info:
    t2v_inputs = {
        "clip": ["3", 0], "vae": ["4", 0],
        "prompt": PROMPT, "width": WIDTH, "height": HEIGHT, "length": FRAMES,
    }
    i2v_spec = object_info.get("MiniMaxH3ImageToVideo", {}).get("input", {})
    if "audio_vae" in i2v_spec.get("required", {}) or "audio_vae" in i2v_spec.get("optional", {}):
        t2v_inputs["audio_vae"] = ["5", 0]
    graph["7"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": t2v_inputs}
elif INFERENCE_MODE in ("reference", "r2v", "t2v", "text_to_video", "director", "char_swap", "v2v"):
    referenced_pictures = [int(match.group(1)) for match in re.finditer(r"<Picture\s+(\d+)>", PROMPT)]
    max_needed = max(referenced_pictures) if referenced_pictures else len(REFERENCE_IMAGE_NAMES)
    while len(REFERENCE_IMAGE_NAMES) < max_needed:
        pad_idx = len(REFERENCE_IMAGE_NAMES) + 1
        pad_name = f"h3_reference_pad_{pad_idx:02d}.png"
        pad_path = input_dir / pad_name
        if REFERENCE_IMAGE_PATHS and Path(REFERENCE_IMAGE_PATHS[-1]).is_file():
            shutil.copyfile(REFERENCE_IMAGE_PATHS[-1], pad_path)
        else:
            Image.new("RGB", (64, 64), (0, 0, 0)).save(pad_path)
        REFERENCE_IMAGE_NAMES.append(pad_path.name)
        REFERENCE_IMAGE_PATHS.append(pad_path)

    picture_inputs = {
        "clip": ["3", 0], "vae": ["4", 0], "audio_vae": ["5", 0],
        "prompt": PROMPT, "width": WIDTH, "height": HEIGHT, "length": FRAMES,
        "ref_image_size": REFERENCE_IMAGE_SIZE,
    }
    for index, image_name in enumerate(REFERENCE_IMAGE_NAMES):
        node_id = str(100 + index)
        graph[node_id] = {"class_type": "LoadImage", "inputs": {"image": image_name}}
        picture_inputs[f"ref_images.ref_image_{index}"] = [node_id, 0]

    if INFERENCE_MODE in ("char_swap", "v2v", "reference", "r2v") and SOURCE_VIDEO_NAME:
        graph["200"] = {"class_type": "LoadVideo", "inputs": {"file": SOURCE_VIDEO_NAME}}
        graph["201"] = {"class_type": "GetVideoComponents", "inputs": {"video": ["200", 0]}}
        picture_inputs["ref_videos.ref_video_0"] = ["201", 0]
        picture_inputs["ref_video_audios.ref_video_audio_0"] = ["201", 1]

    if globals().get("REF_AUDIO_NAME"):
        graph["210"] = {"class_type": "LoadAudio", "inputs": {"audio": REF_AUDIO_NAME}}
        picture_inputs["ref_audios.ref_audio_0"] = ["210", 0]
        print("Audio conditioning: ref_audios.ref_audio_0 <-", REF_AUDIO_NAME)

    picture_count = len(REFERENCE_IMAGE_NAMES)
    if picture_count > 0:
        invalid_pictures = sorted({number for number in referenced_pictures if number < 1 or number > picture_count})
        if invalid_pictures:
            raise ValueError(f"prompt 使用了不存在的圖片標籤：{invalid_pictures}；目前提供 {picture_count} 張。")
    graph["7"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": picture_inputs}
else:
    i2v_inputs = {
        "clip": ["3", 0], "vae": ["4", 0], "first_frame": ["16", 0],
        "prompt": PROMPT, "width": WIDTH, "height": HEIGHT, "length": FRAMES
    }
    i2v_spec = object_info.get("MiniMaxH3ImageToVideo", {}).get("input", {})
    if "audio_vae" in i2v_spec.get("required", {}) or "audio_vae" in i2v_spec.get("optional", {}):
        i2v_inputs["audio_vae"] = ["5", 0]
    graph.update({
        "6": {"class_type": "LoadImage", "inputs": {"image": input_path.name}},
        "16": {"class_type": "ImageScaleToTotalPixels", "inputs": {
            "image": ["6", 0], "upscale_method": FIRST_FRAME_UPSCALE_METHOD,
            "megapixels": FIRST_FRAME_MEGAPIXELS, "resolution_steps": FIRST_FRAME_RESOLUTION_STEPS}},
        "7": {"class_type": "MiniMaxH3ImageToVideo", "inputs": i2v_inputs},
    })
    if globals().get("last_frame_path") and last_frame_path.is_file():
        if "last_frame" in i2v_spec.get("required", {}) or "last_frame" in i2v_spec.get("optional", {}):
            graph["6_last"] = {"class_type": "LoadImage", "inputs": {"image": last_frame_path.name}}
            graph["16_last"] = {"class_type": "ImageScaleToTotalPixels", "inputs": {
                "image": ["6_last", 0], "upscale_method": FIRST_FRAME_UPSCALE_METHOD,
                "megapixels": FIRST_FRAME_MEGAPIXELS, "resolution_steps": FIRST_FRAME_RESOLUTION_STEPS}}
            i2v_inputs["last_frame"] = ["16_last", 0]
            print("FL2VA Keyframe Cuối (last_frame) đã gắn trực tiếp vào MiniMaxH3ImageToVideo:", last_frame_path.name)

if INFERENCE_MODE not in ("longshot", "continuum", "ai_upscale", "face_refine"):
    _sigmas_out = ["11", 0]
    if "H3SigmaRefiner" in object_info and STEPS >= 8:
        _ref_start = 0.7
        graph["11_refiner"] = {
            "class_type": "H3SigmaRefiner",
            "inputs": {
                "sigmas": ["11", 0],
                "extra_steps": 1,
                "start_at_sigma": _ref_start,
                "end_at_sigma": 0.0,
                "spacing": "cosine",
            },
        }
        _sigmas_out = ["11_refiner", 0]
        print(f"⚡ H3SigmaRefiner (+1 bước khử nhiễu chuyển động nhanh, start_at_sigma={_ref_start}) đã kích hoạt!")
    graph.update({
        "8": {"class_type": "BasicGuider", "inputs": {"model": [active_model_node, 0], "conditioning": ["7", 0]}},
        "9": {"class_type": "RandomNoise", "inputs": {"noise_seed": SEED}},
        "10": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": SAMPLER_NAME}},
        "11": {"class_type": "BasicScheduler", "inputs": {"model": [active_model_node, 0], "scheduler": SCHEDULER_NAME, "steps": STEPS, "denoise": DENOISE}},
        "12": {"class_type": "SamplerCustomAdvanced", "inputs": {
            "noise": ["9", 0], "guider": ["8", 0], "sampler": ["10", 0],
            "sigmas": _sigmas_out, "latent_image": ["7", 1]}},
        "13": {"class_type": "VAEDecode", "inputs": {"samples": ["12", 0], "vae": ["4", 0]}},
        "14": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["12", 0], "vae": ["5", 0]}},
        "15": {"class_type": "VHS_VideoCombine", "inputs": {
            "images": ["13", 0], "audio": ["14", 0], "frame_rate": 24,
            "loop_count": 0, "filename_prefix": OUTPUT_PREFIX,
            "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": VIDEO_CRF,
            "save_metadata": True, "trim_to_audio": False,
            "pingpong": False, "save_output": True}},
    })

# === H3 Studio: MiniMaxH3AddGuide anchors (middle keyframes / 1.6s chunk carry / audio pins) ===
try:
    H3_GUIDES = json.loads(os.environ.get("H3_GUIDES", "") or "[]")
except Exception:
    H3_GUIDES = []
if H3_GUIDES and INFERENCE_MODE not in ("longshot", "continuum") and "8" in graph:
    if "MiniMaxH3AddGuide" not in object_info:
        print("WARNING: ComfyUI không có node MiniMaxH3AddGuide -> bỏ qua", len(H3_GUIDES), "mốc guide (cập nhật ComfyUI).")
    else:
        _g_dir = COMFY / "input"
        _g_dir.mkdir(parents=True, exist_ok=True)
        _guide_cond = ["7", 0]
        _n_ok = 0
        for _gi, _gd in enumerate(H3_GUIDES):
            try:
                _gp = Path(str(_gd.get("path", "")))
                if not _gp.is_file():
                    print(f"[guide {_gi}] file không tồn tại: {_gp}")
                    continue
                _kind = str(_gd.get("kind") or "image").lower()
                _fidx = _gd.get("frame")
                if _fidx is None:
                    _fidx = int(round(float(_gd.get("second", 0) or 0) * 24))
                _fidx = int(_fidx)
                _gin = {"positive": _guide_cond, "latent": ["7", 1], "vae": ["4", 0], "audio_vae": ["5", 0]}
                if _kind == "clip":
                    _tail = int(_gd.get("tail_frames", 39) or 39)
                    while _tail > 5 and _tail % 17 != 5:
                        _tail -= 1
                    _fr_dir = _g_dir / f"h3_guide_{_gi}_frames"
                    shutil.rmtree(_fr_dir, ignore_errors=True)
                    _fr_dir.mkdir(parents=True, exist_ok=True)
                    _sseof = f"-{_tail / 24.0 + 0.6:.3f}"
                    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-sseof", _sseof, "-i", str(_gp),
                                    "-vf", f"fps=24,scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT}",
                                    str(_fr_dir / "%05d.png")], check=True)
                    _frames = sorted(_fr_dir.glob("*.png"))
                    if len(_frames) < 5:
                        print(f"[guide {_gi}] clip quá ngắn ({len(_frames)} frame) -> bỏ qua")
                        continue
                    _frames = _frames[-_tail:] if len(_frames) >= _tail else _frames
                    _use = len(_frames)
                    while _use > 5 and _use % 17 != 5:
                        _use -= 1
                    _frames = _frames[-_use:]
                    _clip_name = f"h3_guide_{_gi}.mp4"
                    _lst = _fr_dir / "list.txt"
                    _lst.write_text("".join(f"file '{p.as_posix()}'\nduration {1/24:.6f}\n" for p in _frames), encoding="utf-8")
                    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(_lst),
                                    "-frames:v", str(_use), "-r", "24", "-c:v", "libx264", "-crf", "12", "-pix_fmt", "yuv420p",
                                    str(_g_dir / _clip_name)], check=True)
                    graph[f"g{_gi}_v"] = {"class_type": "LoadVideo", "inputs": {"file": _clip_name}}
                    graph[f"g{_gi}_c"] = {"class_type": "GetVideoComponents", "inputs": {"video": [f"g{_gi}_v", 0]}}
                    _gin["image"] = [f"g{_gi}_c", 0]
                    if _gd.get("with_audio"):
                        _aud_name = f"h3_guide_{_gi}.wav"
                        _ar = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-sseof", f"-{_use / 24.0:.3f}", "-i", str(_gp),
                                              "-vn", "-ac", "2", "-ar", "48000", str(_g_dir / _aud_name)], check=False)
                        if _ar.returncode == 0 and (_g_dir / _aud_name).is_file() and (_g_dir / _aud_name).stat().st_size > 2000:
                            graph[f"g{_gi}_a"] = {"class_type": "LoadAudio", "inputs": {"audio": _aud_name}}
                            _gin["audio"] = [f"g{_gi}_a", 0]
                    print(f"[guide {_gi}] clip {_use} frame{' + audio' if 'audio' in _gin else ''} @ frame {_fidx}")
                elif _kind == "audio":
                    _aud_name = f"h3_guide_{_gi}.wav"
                    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(_gp), "-vn", "-ac", "2", "-ar", "48000",
                                    str(_g_dir / _aud_name)], check=True)
                    graph[f"g{_gi}_a"] = {"class_type": "LoadAudio", "inputs": {"audio": _aud_name}}
                    _gin["audio"] = [f"g{_gi}_a", 0]
                    print(f"[guide {_gi}] audio @ frame {_fidx} ({_fidx / 24:.2f}s)")
                else:
                    _img_name = f"h3_guide_{_gi}.png"
                    with Image.open(_gp) as _im:
                        _im.convert("RGB").save(_g_dir / _img_name)
                    graph[f"g{_gi}_i"] = {"class_type": "LoadImage", "inputs": {"image": _img_name}}
                    _gin["image"] = [f"g{_gi}_i", 0]
                    print(f"[guide {_gi}] keyframe image @ frame {_fidx} ({_fidx / 24:.2f}s)")
                if _fidx >= 0:
                    _max_start = FRAMES - (int(_use) if _kind == "clip" else 1)
                    _fidx = max(0, min(_fidx, _max_start))
                _gin["frame_idx"] = _fidx
                graph[f"g{_gi}"] = {"class_type": "MiniMaxH3AddGuide", "inputs": _gin}
                _guide_cond = [f"g{_gi}", 0]
                _n_ok += 1
            except Exception as _ge:
                print(f"[guide {_gi}] lỗi, bỏ qua: {_ge}")
        graph["8"]["inputs"]["conditioning"] = _guide_cond
        print(f"AddGuide: {_n_ok}/{len(H3_GUIDES)} mốc đã gắn")

# === H3 Studio: SelfLift Progressive-Resolution Sampler (75% Low-Res 0.5x + 3D Latent Lift + 25% Full-Res) ===
if globals().get("USE_SELFLIFT", True) and INFERENCE_MODE not in ("longshot", "continuum", "ai_upscale") and "SelfLiftH3Sampler" in object_info and "ConditioningZeroOut" in object_info and "12" in graph:
    _sl_pos = graph.get("8", {}).get("inputs", {}).get("conditioning", ["7", 0])
    _sl_tstep = max(1, min(STEPS - 1, int(round(STEPS * 0.75))))
    _sl_up_opts = object_info.get("SelfLiftH3Sampler", {}).get("input", {}).get("required", {}).get("upscaler_model", [[]])[0]
    _sl_up_name = "minimax_h3_latent_upscaler_3d_bf16.safetensors"
    if isinstance(_sl_up_opts, list) and _sl_up_name not in _sl_up_opts:
        _h3_cands = [m for m in _sl_up_opts if "h3" in str(m).lower()]
        _sl_up_name = _h3_cands[0] if _h3_cands else "none"
    _sl_rho = 0.0 if _sl_up_name != "none" else 0.5
    graph.pop("8", None)
    graph.pop("11_refiner", None)
    graph["8_neg"] = {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": _sl_pos}}
    graph["12"] = {
        "class_type": "SelfLiftH3Sampler",
        "inputs": {
            "model": [active_model_node, 0],
            "positive": _sl_pos,
            "negative": ["8_neg", 0],
            "vae": ["4", 0],
            "latent_image": ["7", 1],
            "sampler": ["10", 0],
            "sigmas": ["11", 0],
            "seed": SEED,
            "cfg": 1.0,
            "transition_step": _sl_tstep,
            "lowres_scale": 0.5,
            "rho": _sl_rho,
            "w_min": 0.5,
            "w_max": 1.0,
            "upscaler_model": _sl_up_name,
            "highres_tiling": False,
            "upscaler_unload": True,
        },
    }
    print(f"🚀 [SelfLiftH3Sampler] Đã kích hoạt: {_sl_tstep}/{STEPS} bước Low-Res (0.5x) -> Lift ({_sl_up_name}) -> {STEPS - _sl_tstep} bước High-Res (1.0x) + Khử trôi màu Keyframe!")

# === OpenJutsu / Carasibana: H3 FaceRefine (Sửa nét mặt nhỏ / nhân vật đứng xa ở 512x512) ===
if (
    (globals().get("USE_FACE_REFINE", False) or INFERENCE_MODE == "face_refine")
    and "15" in graph
    and {"H3FaceTrackCrop", "H3InjectVideoLatent", "H3FaceStitch", "MiniMaxH3ReferenceToVideo"} <= object_info.keys()
):
    _fr_base_imgs = graph["15"]["inputs"]["images"]
    _det_opts = object_info.get("H3FaceTrackCrop", {}).get("input", {}).get("required", {}).get("detector", [[]])[0]
    _det_val = "face_yolov8m.pt"
    if isinstance(_det_opts, list) and _det_opts:
        for _cand_d in ("bbox/face_yolov8m.pt", "bbox\\face_yolov8m.pt", "face_yolov8m.pt"):
            if _cand_d in _det_opts:
                _det_val = _cand_d
                break
        else:
            _det_val = _det_opts[0]
    _fb_opts = object_info.get("H3FaceTrackCrop", {}).get("input", {}).get("optional", {}).get("fallback_detector", [[]])[0]
    _fb_val = "none"
    if isinstance(_fb_opts, list):
        for _cand_fb in ("segm/person_yolov8m-seg.pt", "segm\\person_yolov8m-seg.pt", "person_yolov8m-seg.pt"):
            if _cand_fb in _fb_opts:
                _fb_val = _cand_fb
                break
    graph["fr_crop"] = {
        "class_type": "H3FaceTrackCrop",
        "inputs": {
            "images": _fr_base_imgs,
            "detector": _det_val,
            "fallback_detector": _fb_val,
            "fallback_head_frac": 0.5,
            "confidence": 0.35,
            "crop_factor": 2.5,
            "canvas_width": 512,
            "canvas_height": 512,
            "canvas_mode": "manual",
            "smooth_window": 21,
            "size_smooth_window": 51,
            "smooth_method": "gaussian",
            "size_mode": "per_frame",
            "identity_track": False,
            "select": "largest_face",
        },
    }
    _fr_prompt = (
        "subject_definitions:\n"
        "<Subject 1> is the person shown in <Picture 1>; their exact facial identity, eyes, nose, lips, skin texture and hair strictly follow <Picture 1>.\n\n"
        "summary:\n"
        "[reference generation] High-detail close-up facial refinement of <Subject 1> from <Picture 1> preserving exact head pose, expression and mouth movement.\n\n"
        "retention_analysis:\n"
        "<Subject 1> (all shots): fully_preserved - exact facial identity and sharp skin detail from <Picture 1>.\n\n"
        "detailed_description:\n"
        "[Shot 1] Photorealistic close-up face of <Subject 1> from <Picture 1>, sharp eyes, natural skin pores, matching lighting and expressions.\n\n"
        "overall_soundscape: Quiet natural room ambience.\n"
        "non_diegetic_music: N/A"
    )
    _fr_ref_in = {
        "clip": ["3", 0],
        "vae": ["4", 0],
        "audio_vae": ["5", 0],
        "prompt": _fr_prompt,
        "width": ["fr_crop", 4],
        "height": ["fr_crop", 5],
        "length": ["fr_crop", 6],
        "ref_image_size": "match",
    }
    if "100" in graph:
        _fr_ref_in["ref_images.ref_image_0"] = ["100", 0]
    elif REFERENCE_IMAGE_NAMES:
        graph["100"] = {"class_type": "LoadImage", "inputs": {"image": REFERENCE_IMAGE_NAMES[0]}}
        _fr_ref_in["ref_images.ref_image_0"] = ["100", 0]
    graph["fr_ref"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": _fr_ref_in}
    graph["fr_inj"] = {
        "class_type": "H3InjectVideoLatent",
        "inputs": {"av_latent": ["fr_ref", 1], "images": ["fr_crop", 0], "vae": ["4", 0]},
    }
    graph["fr_guider"] = {"class_type": "BasicGuider", "inputs": {"model": [active_model_node, 0], "conditioning": ["fr_ref", 0]}}
    graph["fr_sched"] = {
        "class_type": "BasicScheduler",
        "inputs": {"model": [active_model_node, 0], "scheduler": SCHEDULER_NAME, "steps": 4, "denoise": float(os.environ.get("H3_FACE_REFINE_DENOISE", "0.45"))},
    }
    if "9" not in graph:
        graph["9"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": SEED}}
    if "10" not in graph:
        graph["10"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": SAMPLER_NAME}}
    graph["fr_samp"] = {
        "class_type": "SamplerCustomAdvanced",
        "inputs": {
            "noise": ["9", 0],
            "guider": ["fr_guider", 0],
            "sampler": ["10", 0],
            "sigmas": ["fr_sched", 0],
            "latent_image": ["fr_inj", 0],
        },
    }
    graph["fr_dec"] = {"class_type": "VAEDecode", "inputs": {"samples": ["fr_samp", 0], "vae": ["4", 0]}}
    graph["fr_stitch"] = {
        "class_type": "H3FaceStitch",
        "inputs": {
            "base_images": _fr_base_imgs,
            "refined_crops": ["fr_dec", 0],
            "transform": ["fr_crop", 1],
            "paste_region": "face_only",
            "mask_dilation": 16,
            "feather": 6,
            "colour_match": 1.0,
            "blend": 1.0,
            "undetected_frames": "fade_out",
        },
    }
    graph["15"]["inputs"]["images"] = ["fr_stitch", 0]
    print("👤 [H3 FaceRefine] Đã gắn chuỗi Track + Crop 512x512 -> H3InjectVideoLatent (denoise=0.45) -> H3FaceStitch (Sửa nét mặt nhỏ/đứng xa)!")

if AI_UPSCALE and INFERENCE_MODE != "ai_upscale":
    assert "15" in graph and "H3VideoUpsampler" in object_info, "Thiếu node H3VideoUpsampler hoặc output 15 — không được bỏ qua bước upscale."
    _up_in_imgs = graph["15"]["inputs"]["images"]
    _up_in_aud = graph["15"]["inputs"]["audio"]
    graph["14_ai_up"] = {
        "class_type": "H3VideoUpsampler",
        "inputs": {
            "model": ["1", 0],
            "clip": ["3", 0],
            "vae": ["4", 0],
            "images": _up_in_imgs,
            "audio": _up_in_aud,
            "audio_vae": ["5", 0],
            "prompt": "h3upscale, increase resolution and restore fine details while preserving the original content and visual style.",
            "upscale_model": "minimax_h3_latent_upscaler_3d_bf16.safetensors",
            "megapixels": 2.0,
            "denoise": 0.1,
            "lora": "h3upscale_v2_x2.safetensors",
            "lora_strength": 1.0,
            "tile_tokens": 180000,
            "sampler_name": "euler",
        },
    }
    graph["15"]["inputs"]["images"] = ["14_ai_up", 0]
    graph["15"]["inputs"]["audio"] = ["14_ai_up", 1]
    print("💎 Đã gắn node H3VideoUpsampler (AI Latent Upscale x2 + h3upscale_v2_x2 LoRA) vào cuối quy trình!")

# === OpenJutsu Fix Colours: ImageDetailContrast (-0.20) + Orion4D DCTL ToneMapper (H3 Soft Reinhard) ===
if "15" in graph and globals().get("USE_COLOR_GRADE", True) and os.environ.get("H3_COLOR_GRADE", "1") != "0":
    if "ImageDetailContrast" in object_info:
        _cg_in = graph["15"]["inputs"]["images"]
        graph["14_oj_detail"] = {
            "class_type": "ImageDetailContrast",
            "inputs": {
                "image": _cg_in,
                "detail_contrast": -0.20,
                "blur_radius": 24,
                "blur_sigma": 8.0,
                "temporal_mode": "temporal_smooth",
                "chunk_size": 8,
            },
        }
        graph["15"]["inputs"]["images"] = ["14_oj_detail", 0]
    if "Orion4D_FXMax_DCTLToneMapper" in object_info:
        _tm_in = graph["15"]["inputs"]["images"]
        _oj_raw_settings = json.dumps({
            "node_type": "color_fx_dctl_tone_mapper",
            "preset": "Openjutsu",
            "settings": {
                "mode": "Reinhard",
                "exposure": 0.09,
                "contrast": 0.84,
                "pivot": 0.69,
                "highlight_rolloff": 0.27,
                "shadow_lift": 0.15,
                "black_floor": 0.03,
                "saturation": 1.05,
                "strength": 0.20,
                "preserve_luminosity": False,
                "clamp_output": True,
            },
        })
        graph["14_oj_tone"] = {
            "class_type": "Orion4D_FXMax_DCTLToneMapper",
            "inputs": {
                "image": _tm_in,
                "preset": "Openjutsu",
                "raw_settings": _oj_raw_settings,
            },
        }
        graph["15"]["inputs"]["images"] = ["14_oj_tone", 0]
    if "14_oj_detail" in graph or "14_oj_tone" in graph:
        print("🎨 [OpenJutsu H3 Soft] Đã bật hậu kỳ màu điện ảnh: ImageDetailContrast (-0.20 temporal_smooth) + Orion4D DCTL ToneMapper (Reinhard H3 Soft)!")

if "15" in graph and "FrameInterpolationModelLoader" in object_info and "FrameInterpolate" in object_info and os.environ.get("H3_RIFE_48FPS", "1") != "0":
    _rife_in = graph["15"]["inputs"]["images"]
    graph["14_rife_loader"] = {
        "class_type": "FrameInterpolationModelLoader",
        "inputs": {"model_name": "rife_v4.26_heavy.safetensors"},
    }
    graph["14_rife_interp"] = {
        "class_type": "FrameInterpolate",
        "inputs": {
            "interp_model": ["14_rife_loader", 0],
            "images": _rife_in,
            "multiplier": 2,
        },
    }
    graph["15"]["inputs"]["images"] = ["14_rife_interp", 0]
    graph["15"]["inputs"]["frame_rate"] = 48
    print("🎞️ Đã gắn RIFE v4.26 Heavy (Comfy-Org/Frame-Interpolation) nội suy 24fps -> 48fps trực tiếp trên GPU Colab!")

for node in graph.values():
    spec = object_info[node["class_type"]]["input"]
    known = set(spec.get("required", {})) | set(spec.get("optional", {}))
    if node["class_type"] == "VHS_VideoCombine":
        format_info = spec["required"]["format"][1]["formats"]
        format_name = node["inputs"]["format"]
        if format_name not in format_info:
            raise ValueError(f"VideoHelperSuite 不支援輸出格式：{format_name}")
        known.update(widget[0] for widget in format_info[format_name])
    dynamic_reference_inputs = {
        key for key in node["inputs"]
        if (
            (node["class_type"] in ("MiniMaxH3ReferenceToVideo", "MiniMaxH3LongShot")
             and re.fullmatch(r"(ref_images\.ref_image_\d+|ref_videos\.ref_video_\d+|ref_video_audios\.ref_video_audio_\d+|ref_audios\.ref_audio_\d+)", key))
            or ("." in key and key.split(".", 1)[0] in known)
        )
    }
    unexpected = set(node["inputs"]) - known - dynamic_reference_inputs
    if unexpected:
        raise ValueError(f"{node['class_type']} 的參數已變更：{sorted(unexpected)}；請更新 notebook。")
result = requests.post(BASE_URL + "/prompt", json={"prompt": graph, "client_id": "h3-colab"}, timeout=60)
if not result.ok:
    raise RuntimeError(f"提交失敗 HTTP {result.status_code}: {result.text}")
job = result.json()
if job.get("node_errors"):
    raise RuntimeError("節點驗證失敗：" + json.dumps(job["node_errors"], ensure_ascii=False, indent=2))
prompt_id = job["prompt_id"]
print("已提交推論：", prompt_id)


## 5. 等待、檢查並儲存 MP4

大型模型首次載入可能需要數分鐘或更久。輸出 cell 最多等待兩小時，檢查 MP4 同時有影片與音訊串流。網頁模式會顯示預覽並提供下載；本機 CLI 模式會複製到 `/content/minimax_h3_output.mp4`，讓 launcher 下載到本機。音訊串流存在不代表音訊內容一定理想；若畫面或聲音不如預期，先用相同 seed 比較 prompt、時長、步數與模型版本，並查看 `/content/comfyui_h3.log`。


In [ ]:
job_timeout_seconds = min(float(os.environ.get("H3_JOB_TIMEOUT_SECONDS", "2400")), 3600.0)
assert math.isfinite(job_timeout_seconds) and job_timeout_seconds > 0, "H3_JOB_TIMEOUT_SECONDS 必須是正數。"
deadline = time.time() + job_timeout_seconds
last_report = 0
empty_queue_count = 0
while time.time() < deadline:
    if 'COMFY_PROCESS' in globals() and COMFY_PROCESS.poll() is not None:
        raise RuntimeError(f"ComfyUI 已退出；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-5000:])
    history = requests.get(BASE_URL + f"/history/{prompt_id}", timeout=30).json()
    if prompt_id in history:
        run = history[prompt_id]
        if run.get("status", {}).get("status_str") == "error":
            raise RuntimeError("推論失敗：" + json.dumps(run.get("status", {}), ensure_ascii=False, indent=2) + f"\n日誌：{LOG_PATH}\n" + (LOG_PATH.read_text()[-3000:] if LOG_PATH.is_file() else ""))
        outputs = run.get("outputs", {})
        if "15" in outputs:
            break
        if run.get("status", {}).get("status_str") == "success":
            raise RuntimeError("任務已完成但沒有 MP4 輸出：" + json.dumps(outputs, ensure_ascii=False))
    else:
        try:
            q_info = requests.get(BASE_URL + "/queue", timeout=15).json()
            q_ids = [item[1] for item in q_info.get("queue_running", []) + q_info.get("queue_pending", []) if isinstance(item, list) and len(item) > 1]
            if prompt_id not in q_ids:
                empty_queue_count += 1
                if empty_queue_count >= 3:
                    log_tail = LOG_PATH.read_text()[-4000:] if LOG_PATH.is_file() else ""
                    raise RuntimeError(f"ComfyUI 任務已中斷（不在佇列中，可能發生 OOM 或內部錯誤）：\n{log_tail}")
            else:
                empty_queue_count = 0
        except RuntimeError:
            raise
        except Exception:
            pass
    if time.time() - last_report >= 30:
        log_snip = ""
        if LOG_PATH.is_file():
            lines = [ln.strip() for ln in LOG_PATH.read_text(errors="ignore").splitlines() if ln.strip()]
            if lines:
                log_snip = " | " + lines[-1][-120:]
        print("仍在生成影片…" + log_snip)
        last_report = time.time()
    time.sleep(10)
else:
    raise TimeoutError(f"等待推論超過兩小時；可查詢 /history/{prompt_id}，日誌：{LOG_PATH}")

filenames = outputs["15"].get("gifs", [])
if not filenames:
    raise RuntimeError("VideoHelperSuite 未回傳 MP4 輸出；請查看 " + str(LOG_PATH))
video_info = next((item for item in filenames if item["filename"].endswith(".mp4")), None)
if video_info is None:
    raise RuntimeError("找不到 MP4；輸出：" + str(filenames))
video_path = COMFY / "output" / video_info.get("subfolder", "") / video_info["filename"]
assert video_path.is_file() and video_path.stat().st_size > 0, f"輸出檔不存在：{video_path}"
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "stream=codec_type",
     "-of", "json", str(video_path)], capture_output=True, text=True, check=True)
streams = {item["codec_type"] for item in json.loads(probe.stdout).get("streams", [])}
assert {"video", "audio"} <= streams, f"MP4 缺少影片或音訊串流：{streams}"
print("完成：", video_path, f"({video_path.stat().st_size / 2**20:.1f} MiB，含影片與音訊)")

# 混音外部音訊軌（若有上傳 MP3 / WAV 參照音檔）
AUDIO_SRC = os.environ.get("H3_AUDIO_PATH")
if AUDIO_SRC and Path(AUDIO_SRC).is_file():
    muxed_path = video_path.with_name(video_path.stem + "_muxed.mp4")
    try:
        subprocess.run([
            "ffmpeg", "-y", "-i", str(video_path), "-i", str(AUDIO_SRC),
            "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
            "-map", "0:v:0", "-map", "1:a:0", "-shortest", str(muxed_path)
        ], check=True, capture_output=True)
        if muxed_path.is_file() and muxed_path.stat().st_size > 0:
            video_path = muxed_path
            print("已成功將參照音訊軌 (MP3) 混音至影片中：", video_path)
    except Exception as e:
        print("音訊混音略過：", e)

if CLI_MODE:
    CLI_OUTPUT_PATH = Path(os.environ.get("H3_OUTPUT_PATH", "/content/minimax_h3_output.mp4"))
    CLI_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(video_path, CLI_OUTPUT_PATH)
    print("COLAB_CLI_OUTPUT_PATH:", CLI_OUTPUT_PATH, flush=True)
    _multi_file = os.environ.get("H3_MULTI_JOBS_FILE", "").strip()
    if _multi_file and Path(_multi_file).is_file():
        _extra_jobs = json.loads(Path(_multi_file).read_text(encoding="utf-8"))
        for _m_idx, _m_job in enumerate(_extra_jobs, start=1):
            try:
                print(f"\n=== [CHẠY LIÊN TIẾP TRONG CÙNG 1 PHIÊN VRAM] Đoạn {_m_idx + 1}/{len(_extra_jobs) + 1}: {_m_job.get('title', _m_job.get('id', ''))} ===", flush=True)
                _m_mode = str(_m_job.get("mode") or INFERENCE_MODE).strip().lower()
                _m_prompt = _format_h3_prompt(_m_job.get("prompt", ""), _m_mode)
                _m_w = int(_m_job.get("width") or WIDTH)
                _m_h = int(_m_job.get("height") or HEIGHT)
                _m_dur = float(_m_job.get("duration_seconds") or DURATION_SECONDS)
                _m_req_f = max(5, round(_m_dur * 24))
                _m_frames = _m_req_f + (5 - _m_req_f % 17) % 17
                if "7" in graph and isinstance(graph["7"].get("inputs"), dict):
                    graph["7"]["inputs"]["prompt"] = _m_prompt
                    graph["7"]["inputs"]["width"] = _m_w
                    graph["7"]["inputs"]["height"] = _m_h
                    graph["7"]["inputs"]["length"] = _m_frames
                    if "remote_refs" in _m_job and graph["7"].get("class_type") == "MiniMaxH3ReferenceToVideo":
                        for _old_i in range(20):
                            graph.pop(str(100 + _old_i), None)
                            graph["7"]["inputs"].pop(f"ref_images.ref_image_{_old_i}", None)
                        _m_rrefs = _m_job.get("remote_refs") or []
                        if _m_mode not in ("t2v", "text_to_video") and _m_rrefs:
                            _m_in_dir = COMFY / "input"
                            _m_in_dir.mkdir(parents=True, exist_ok=True)
                            _m_names = []
                            _m_paths = []
                            for _ri, _rp in enumerate(_m_rrefs, start=1):
                                _src_p = Path(_rp)
                                if _src_p.is_file():
                                    _dst_p = _m_in_dir / f"h3_m{_m_idx}_ref_{_ri:02d}.png"
                                    with Image.open(_src_p) as _op_im:
                                        _op_im.convert("RGB").save(_dst_p)
                                    _m_names.append(_dst_p.name)
                                    _m_paths.append(_dst_p)
                            _m_pic_nums = [int(_mt.group(1)) for _mt in re.finditer(r"<Picture\s+(\d+)>", _m_prompt)]
                            _m_needed = max(_m_pic_nums) if _m_pic_nums else len(_m_names)
                            while len(_m_names) < _m_needed:
                                _pad_i = len(_m_names) + 1
                                _pad_p = _m_in_dir / f"h3_m{_m_idx}_pad_{_pad_i:02d}.png"
                                if _m_paths and _m_paths[-1].is_file():
                                    shutil.copyfile(_m_paths[-1], _pad_p)
                                else:
                                    Image.new("RGB", (64, 64), (0, 0, 0)).save(_pad_p)
                                _m_names.append(_pad_p.name)
                                _m_paths.append(_pad_p)
                            for _ni, _nm in enumerate(_m_names):
                                _nid = str(100 + _ni)
                                graph[_nid] = {"class_type": "LoadImage", "inputs": {"image": _nm}}
                                graph["7"]["inputs"][f"ref_images.ref_image_{_ni}"] = [_nid, 0]
                _next_seed = int(_m_job.get("seed") or (SEED + _m_idx))
                if "9" in graph and "noise_seed" in graph["9"].get("inputs", {}):
                    graph["9"]["inputs"]["noise_seed"] = _next_seed
                if "12" in graph and "seed" in graph["12"].get("inputs", {}):
                    graph["12"]["inputs"]["seed"] = _next_seed
                if "15" in graph and "filename_prefix" in graph["15"].get("inputs", {}):
                    graph["15"]["inputs"]["filename_prefix"] = f"{OUTPUT_PREFIX}_part{_m_idx+1:02d}"
                if _m_job.get("chain_last_frame") and video_path and Path(video_path).is_file():
                    _lf_name = f"h3_chain_lf_{_m_idx:02d}.png"
                    _lf_dest = COMFY / "input" / _lf_name
                    subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-sseof", "-0.08", "-i", str(video_path), "-frames:v", "1", "-q:v", "1", str(_lf_dest)], check=False)
                    if _lf_dest.is_file() and _lf_dest.stat().st_size > 500:
                        # Drift-Control Anti-Burn: cân bằng trung bình màu & chống cháy sáng so với ảnh gốc <Picture 1>
                        try:
                            import numpy as _np
                            _anchor_p = REFERENCE_IMAGE_PATHS[0] if REFERENCE_IMAGE_PATHS else None
                            if _anchor_p and Path(_anchor_p).is_file():
                                _im_lf = _np.asarray(Image.open(_lf_dest).convert("RGB"), dtype=_np.float32)
                                _im_rf = _np.asarray(Image.open(_anchor_p).convert("RGB"), dtype=_np.float32)
                                for _ch in range(3):
                                    _m_lf, _s_lf = float(_im_lf[:, :, _ch].mean()), max(float(_im_lf[:, :, _ch].std()), 1.0)
                                    _m_rf, _s_rf = float(_im_rf[:, :, _ch].mean()), max(float(_im_rf[:, :, _ch].std()), 1.0)
                                    _t_m = _m_lf * 0.70 + _m_rf * 0.30
                                    _t_s = _s_lf * 0.75 + _s_rf * 0.25
                                    _im_lf[:, :, _ch] = (_im_lf[:, :, _ch] - _m_lf) * (_t_s / _s_lf) + _t_m
                                Image.fromarray(_np.clip(_im_lf, 2.0, 253.0).round().astype(_np.uint8)).save(_lf_dest)
                                print(f"🎨 [Drift-Control Anti-Burn] Đã khóa cân bằng màu & chống cháy sáng cho {_lf_name}", flush=True)
                        except Exception as _ab_e:
                            pass
                        if _m_mode == "first_frame" and "6" in graph:
                            graph["6"]["inputs"]["image"] = _lf_name
                            print(f"[Chain] Đã gắn frame cuối đoạn trước vào first_frame: {_lf_name}", flush=True)
                        elif ("8" in graph or "12" in graph) and "MiniMaxH3AddGuide" in object_info:
                            graph["g_chain_0"] = {"class_type": "LoadImage", "inputs": {"image": _lf_name}}
                            graph["g_chain_node"] = {
                                "class_type": "MiniMaxH3AddGuide",
                                "inputs": {
                                    "positive": ["7", 0],
                                    "latent": ["7", 1],
                                    "vae": ["4", 0],
                                    "audio_vae": ["5", 0],
                                    "image": ["g_chain_0", 0],
                                    "frame_idx": 0,
                                },
                            }
                            if "8" in graph and "conditioning" in graph["8"].get("inputs", {}):
                                graph["8"]["inputs"]["conditioning"] = ["g_chain_node", 0]
                            if "12" in graph and "positive" in graph["12"].get("inputs", {}):
                                graph["12"]["inputs"]["positive"] = ["g_chain_node", 0]
                            if "8_neg" in graph and "conditioning" in graph["8_neg"].get("inputs", {}):
                                graph["8_neg"]["inputs"]["conditioning"] = ["g_chain_node", 0]
                            print(f"[Chain] Đã khóa nối tiếp frame cuối đoạn trước tại frame 0 (MiniMaxH3AddGuide + SelfLift): {_lf_name}", flush=True)
                else:
                    graph.pop("g_chain_0", None)
                    graph.pop("g_chain_node", None)
                    if "8" in graph and "conditioning" in graph["8"].get("inputs", {}):
                        graph["8"]["inputs"]["conditioning"] = ["7", 0]
                    if "12" in graph and "positive" in graph["12"].get("inputs", {}):
                        graph["12"]["inputs"]["positive"] = ["7", 0]
                    if "8_neg" in graph and "conditioning" in graph["8_neg"].get("inputs", {}):
                        graph["8_neg"]["inputs"]["conditioning"] = ["7", 0]
                    print("[Chain] Tạo độc lập video tiếp theo trong cùng phiên VRAM", flush=True)
                _res_m = requests.post(BASE_URL + "/prompt", json={"prompt": graph, "client_id": "h3-colab"}, timeout=60)
                if not _res_m.ok:
                    raise RuntimeError(f"Multi-clip submit failed HTTP {_res_m.status_code}: {_res_m.text}")
                _job_m = _res_m.json()
                _pid_m = _job_m["prompt_id"]
                print("已提交推論：", _pid_m, flush=True)
                _dl_m = time.time() + job_timeout_seconds
                _last_rep_m = 0
                while time.time() < _dl_m:
                    try:
                        _hist_m = requests.get(BASE_URL + f"/history/{_pid_m}", timeout=30).json()
                    except Exception:
                        _hist_m = {}
                    if _pid_m in _hist_m:
                        _run_m = _hist_m[_pid_m]
                        if _run_m.get("status", {}).get("status_str") == "error":
                            raise RuntimeError("Multi-clip inference error: " + json.dumps(_run_m.get("status", {}), ensure_ascii=False))
                        _outs_m = _run_m.get("outputs", {})
                        if "15" in _outs_m:
                            break
                    if time.time() - _last_rep_m >= 10:
                        print(f"仍在生成影片… (Đoạn {_m_idx+1}/{len(_extra_jobs)+1})", flush=True)
                        _last_rep_m = time.time()
                    time.sleep(4)
                else:
                    raise TimeoutError(f"Timeout waiting for multi-clip {_pid_m}")
                _fnames_m = _outs_m["15"].get("gifs", [])
                _vinfo_m = next((it for it in _fnames_m if it["filename"].endswith(".mp4")), None)
                assert _vinfo_m is not None, f"No MP4 output for multi-clip {_m_idx}"
                video_path = COMFY / "output" / _vinfo_m.get("subfolder", "") / _vinfo_m["filename"]
                _rem_out_m = Path(_m_job["remote_output"])
                _rem_out_m.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(video_path, _rem_out_m)
                print(f"H3_MULTI_CLIP_READY:{_m_idx}:{_rem_out_m}", flush=True)
            except Exception as _m_err:
                print(f"H3_MULTI_CLIP_ERROR:{_m_idx}:{_m_err}", flush=True)
else:
    from IPython.display import Video, display
    from google.colab import files

    display(Video(str(video_path), embed=True))
    files.download(str(video_path))


## 常見調整與限制

- **想讓切鏡更明顯：**在新 shot 描述中寫清楚「the camera cuts to」後的新景別、視角和主體狀態。只改一點點角度時，改寫成同一 shot 的鏡頭運動通常較自然。切鏡時間由模型理解，不能保證影格精準。
- **角色在不同 shot 變樣：**每一鏡重申同一角色與場景的關鍵特徵；Shot 1 的首幀只能錨定開頭，後面仍靠文字與模型維持一致。
- **GPU 記憶體不足：**降低 `WIDTH`、`HEIGHT` 或 `DURATION_SECONDS`，再從第 3 節重跑；保持寬高都是 32 的倍數。也可先使用 5 秒、一到兩鏡確認流程。
- **音訊失真或沉默：**檢查環境聲/對白描述、取樣設定與日誌。模型同時生成影片和音訊，MP4 串流檢查只能確認音訊軌存在。
- **想提供多張語意參照圖片：**切換到 `reference` 模式；CLI launcher 會自動切換，最多可上傳 9 張，並在完整 prompt 中以 `<Picture N>` 標註。這些圖片提供語意條件，不會自動固定在不同影片時間；需要時間軸 keyframe 時，還需接上官方 `Add Guide for MiniMax H3` 節點。
- **想讀本機 prompt.txt：**使用 CLI `--prompt` 指定 UTF-8 文字檔；內容會直接成為 Ref2VA prompt，不會由 `SHOTS` 重新組裝。
- **與原 workflow 畫面不同：**本 notebook 不裝原 workflow 的可選注意力補丁、Spectrum、預覽和除錯節點，並使用官方文字編碼器。完整原流程與本版的輸出可能不同。
